# 1\.2\.5 Standardize and Harmonize Weather

This notebook brings weather into the same project contract as the mobility layers without pretending that four stations are the same thing as native Taxi Zone activity\. The job here is to standardize the curated hourly weather export onto the shared temporal framework, keep a compact retained feature set, and build the zone\-level contextual handoff that later notebooks can join when weather is actually useful\. In practice, this is the bridge between the staged weather sidecar from \`1\.1\.7\` and the downstream mobility products that need weather as context rather than as a standalone modality\.

Before we start the first chapter, we set the notebook\-level constants, paths, rebuild toggles, and a few small temporal helpers that keep the workflow aligned with \`1\.2\.1\`, \`1\.2\.4\`, and \`1\.3\.1\`\. The aim is to keep the weather pipeline explicit and lightweight: one curated weather input, one consistent temporal contract, one staged set of downstream output paths, and just enough helper logic to avoid duplicating temporal definitions later in the notebook\.

In [1]:
%pip install plotly==6.8.0


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

from IPython.display import display

import geopandas as gpd
from shapely.geometry import Point

In [3]:
# ---------------------------------------------------------------------
# Notebook-level constants, paths, toggles, and helpers
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

STUDY_START_DATE = pd.Timestamp("2023-01-01")
STUDY_END_DATE = pd.Timestamp("2026-03-31")
CONGESTION_PRICING_START_DATE = pd.Timestamp("2025-01-05")

WEATHER_CURATED_INPUT_PATH = (
    PIPELINE_DATA_DIR
    / "1.1.7.staged_weather"
    / "weather_hourly_observations.parquet"
)

WEATHER_HARMONIZED_OUTPUT_DIR = (
    PIPELINE_DATA_DIR / "1.2.5.harmonized_weather"
)
WEATHER_HARMONIZED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WEATHER_TEMPORAL_OUTPUT_PATH = (
    WEATHER_HARMONIZED_OUTPUT_DIR / "weather_station_hourly_temporal.parquet"
)
WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH = (
    WEATHER_HARMONIZED_OUTPUT_DIR / "weather_station_hourly_canonical.parquet"
)
WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH = (
    WEATHER_HARMONIZED_OUTPUT_DIR / "weather_station_temporal_bucket.parquet"
)
WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH = (
    WEATHER_HARMONIZED_OUTPUT_DIR / "taxi_zone_weather_assignment_weights.parquet"
)
WEATHER_ZONE_CONTEXT_OUTPUT_PATH = (
    WEATHER_HARMONIZED_OUTPUT_DIR / "weather_taxi_zone_context.parquet"
)

REBUILD_WEATHER_TEMPORAL = False
REBUILD_WEATHER_CANONICAL_STATION_HOUR = False
REBUILD_WEATHER_STATION_BUCKET = False
REBUILD_WEATHER_ZONE_ASSIGNMENT_WEIGHTS = False
REBUILD_WEATHER_ZONE_CONTEXT = False
RUN_WEATHER_VALIDATION = True

RETAINED_WEATHER_MEASURE_COLUMNS = [
    "temperature",
    "precipitation",
    "visibility",
    "wind_speed",
    "wind_gust",
    "relative_humidity",
    "ceiling_height",
    "pressure_3hr_change",
]

WEATHER_DISTANCE_WEIGHT_EPSILON_MILES = 0.1

DAYPART_ORDER = {
    "overnight": 1,
    "am_peak": 2,
    "midday": 3,
    "pm_peak": 4,
    "evening": 5,
}

TEMPORAL_BUCKET_ORDER = {
    "weekday_overnight": 1,
    "weekday_am_peak": 2,
    "weekday_midday": 3,
    "weekday_pm_peak": 4,
    "weekday_evening": 5,
    "weekend_overnight": 6,
    "weekend_am_peak": 7,
    "weekend_midday": 8,
    "weekend_pm_peak": 9,
    "weekend_evening": 10,
}

VALID_TEMPORAL_BUCKETS = set(TEMPORAL_BUCKET_ORDER)

weather_setup_summary_df = pd.DataFrame(
    {
        "setting": [
            "weather_curated_input_path",
            "weather_temporal_output_path",
            "weather_canonical_station_hour_output_path",
            "weather_station_temporal_bucket_output_path",
            "weather_zone_assignment_weights_output_path",
            "weather_zone_context_output_path",
            "rebuild_weather_temporal",
            "rebuild_weather_canonical_station_hour",
            "rebuild_weather_station_bucket",
            "rebuild_weather_zone_assignment_weights",
            "rebuild_weather_zone_context",
            "run_weather_validation",
            "weather_distance_weight_epsilon_miles",
            "retained_weather_measure_columns",
        ],
        "value": [
            str(WEATHER_CURATED_INPUT_PATH),
            str(WEATHER_TEMPORAL_OUTPUT_PATH),
            str(WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH),
            str(WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH),
            str(WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH),
            str(WEATHER_ZONE_CONTEXT_OUTPUT_PATH),
            REBUILD_WEATHER_TEMPORAL,
            REBUILD_WEATHER_CANONICAL_STATION_HOUR,
            REBUILD_WEATHER_STATION_BUCKET,
            REBUILD_WEATHER_ZONE_ASSIGNMENT_WEIGHTS,
            REBUILD_WEATHER_ZONE_CONTEXT,
            RUN_WEATHER_VALIDATION,
            WEATHER_DISTANCE_WEIGHT_EPSILON_MILES,
            ", ".join(RETAINED_WEATHER_MEASURE_COLUMNS),
        ],
    }
)

display(weather_setup_summary_df)

,setting,value
0,weather_curated_input_path,pipeline_data/1.1.7.staged_weather/weather_hou...
1,weather_temporal_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
2,weather_canonical_station_hour_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
3,weather_station_temporal_bucket_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
4,weather_zone_assignment_weights_output_path,pipeline_data/1.2.5.harmonized_weather/taxi_zo...
5,weather_zone_context_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
6,rebuild_weather_temporal,False
7,rebuild_weather_canonical_station_hour,False
8,rebuild_weather_station_bucket,False
9,rebuild_weather_zone_assignment_weights,False


In [4]:
def print_step(message: str) -> None:
    """Emit compact progress messages for notebook execution."""
    print(f"[1.2.5] {message}", flush=True)



def should_build(output_path: Path, rebuild_flag: bool) -> bool:
    """Return True when a weather output should be rebuilt."""
    return rebuild_flag or not output_path.exists()



def assign_daypart(hour):
    """Assign the project-standard daypart from an hour-of-day value."""
    if pd.isna(hour):
        return np.nan

    hour = int(hour)
    if 0 <= hour < 6:
        return "overnight"
    if 6 <= hour < 10:
        return "am_peak"
    if 10 <= hour < 16:
        return "midday"
    if 16 <= hour < 20:
        return "pm_peak"
    if 20 <= hour < 24:
        return "evening"
    return np.nan



def assign_temporal_bucket(day_of_week, hour):
    """Assign the canonical temporal bucket label."""
    daypart = assign_daypart(hour)
    if pd.isna(daypart) or pd.isna(day_of_week):
        return np.nan

    day_type = "weekend" if day_of_week in ["Saturday", "Sunday"] else "weekday"
    return f"{day_type}_{daypart}"



def add_shared_temporal_fields(df: pd.DataFrame) -> pd.DataFrame:
    """Add the shared project temporal contract fields to weather rows."""
    df = df.copy()
    df["observation_datetime"] = pd.to_datetime(df["observation_datetime"], errors="coerce")
    df = df[df["observation_datetime"].notna()].copy()

    df["date"] = df["observation_datetime"].dt.normalize()
    df = df[df["date"].between(STUDY_START_DATE, STUDY_END_DATE)].copy()

    df["year"] = df["observation_datetime"].dt.year.astype("int64")
    df["month"] = df["observation_datetime"].dt.month.astype("int64")
    df["day_of_week"] = df["observation_datetime"].dt.day_name()
    df["hour"] = df["observation_datetime"].dt.hour.astype("int64")
    df["daypart"] = df["hour"].apply(assign_daypart)
    df["daypart_order"] = df["daypart"].map(DAYPART_ORDER).astype("Int64")
    df["temporal_bucket"] = df.apply(
        lambda row: assign_temporal_bucket(row["day_of_week"], row["hour"]),
        axis=1,
    )
    df["temporal_bucket_order"] = df["temporal_bucket"].map(TEMPORAL_BUCKET_ORDER).astype("Int64")
    df["observation_sequence_id"] = (
        (df["date"] - STUDY_START_DATE).dt.days * len(DAYPART_ORDER)
        + df["daypart_order"]
    ).astype("Int64")
    df["pre_post_cp"] = np.where(
        df["date"] >= CONGESTION_PRICING_START_DATE,
        "post_cp",
        "pre_cp",
    )
    df["temporal_grain_type"] = "native_hourly_station_observation"
    df["date_assignment_method"] = "derived_from_native_station_timestamp"
    return df


print_step("Weather harmonization setup loaded.")

display(weather_setup_summary_df)

[1.2.5] Weather harmonization setup loaded.


,setting,value
0,weather_curated_input_path,pipeline_data/1.1.7.staged_weather/weather_hou...
1,weather_temporal_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
2,weather_canonical_station_hour_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
3,weather_station_temporal_bucket_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
4,weather_zone_assignment_weights_output_path,pipeline_data/1.2.5.harmonized_weather/taxi_zo...
5,weather_zone_context_output_path,pipeline_data/1.2.5.harmonized_weather/weather...
6,rebuild_weather_temporal,False
7,rebuild_weather_canonical_station_hour,False
8,rebuild_weather_station_bucket,False
9,rebuild_weather_zone_assignment_weights,False


## 1\.2\.5\.1 Load curated staged weather and add shared temporal fields

This first section moves the curated weather export out of source\-loading mode and into the shared project time contract\. The main job is straightforward: load the retained hourly weather observations written by \`1\.1\.7\`, derive the same date, daypart, temporal\-bucket, ordering, and congestion\-pricing fields used elsewhere in the mobility pipeline, and write a temporally standardized station\-hour weather table that downstream chapters can trust\.

We begin by loading the curated hourly weather sidecar and checking that the expected retained fields are available before adding any new contract fields\. Then we apply the shared temporal definitions already used across the project, validate that the resulting station\-hour weather rows stay inside the study period, and save the standardized output that later sections will aggregate and spatially evaluate\.

In [5]:
# ---------------------------------------------------------------------
# 1.2.5.1 Load curated staged weather and add shared temporal fields
# ---------------------------------------------------------------------

WEATHER_TEMPORAL_REQUIRED_COLUMNS = [
    "weather_station_id",
    "weather_station_name",
    "observation_datetime",
    *RETAINED_WEATHER_MEASURE_COLUMNS,
]

if should_build(WEATHER_TEMPORAL_OUTPUT_PATH, REBUILD_WEATHER_TEMPORAL):
    print_step("Loading curated staged weather observations.")
    weather_curated_df = pd.read_parquet(WEATHER_CURATED_INPUT_PATH)

    missing_required_columns = [
        column
        for column in WEATHER_TEMPORAL_REQUIRED_COLUMNS
        if column not in weather_curated_df.columns
    ]
    if missing_required_columns:
        raise ValueError(
            f"Curated weather input is missing required columns: {missing_required_columns}"
        )

    print_step(
        f"Applying shared temporal fields to {len(weather_curated_df):,} curated weather rows."
    )
    weather_temporal_df = add_shared_temporal_fields(weather_curated_df)

    if RUN_WEATHER_VALIDATION:
        invalid_temporal_bucket_rows = (
            ~weather_temporal_df["temporal_bucket"].isin(VALID_TEMPORAL_BUCKETS)
        ).sum()
        if invalid_temporal_bucket_rows != 0:
            raise ValueError(
                f"Weather temporal output has {invalid_temporal_bucket_rows:,} invalid temporal_bucket rows."
            )

        if weather_temporal_df["observation_sequence_id"].isna().any():
            raise ValueError("Weather temporal output contains null observation_sequence_id values.")

        if weather_temporal_df["pre_post_cp"].isna().any():
            raise ValueError("Weather temporal output contains null pre_post_cp values.")

    weather_temporal_df.to_parquet(WEATHER_TEMPORAL_OUTPUT_PATH, index=False)
    print_step(f"Wrote temporal-standardized weather output: {WEATHER_TEMPORAL_OUTPUT_PATH}")

else:
    print_step("Loading existing temporal-standardized weather output.")
    weather_temporal_df = pd.read_parquet(WEATHER_TEMPORAL_OUTPUT_PATH)

weather_temporal_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "station_count",
            "min_date",
            "max_date",
            "distinct_dates",
            "distinct_temporal_buckets",
            "missing_observation_datetime_rows",
        ],
        "value": [
            len(weather_temporal_df),
            weather_temporal_df["weather_station_id"].nunique(),
            weather_temporal_df["date"].min(),
            weather_temporal_df["date"].max(),
            weather_temporal_df["date"].nunique(),
            weather_temporal_df["temporal_bucket"].nunique(),
            weather_temporal_df["observation_datetime"].isna().sum(),
        ],
    }
)

display(weather_temporal_summary_df)

display(
    weather_temporal_df[
        [
            "weather_station_id",
            "weather_station_name",
            "observation_datetime",
            "date",
            "day_of_week",
            "hour",
            "daypart",
            "temporal_bucket",
            "temporal_bucket_order",
            "observation_sequence_id",
            "pre_post_cp",
        ]
    ].head(10)
)

[1.2.5] Loading existing temporal-standardized weather output.


,metric,value
0,row_count,221939
1,station_count,4
2,min_date,2023-01-01 00:00:00
3,max_date,2026-03-31 00:00:00
4,distinct_dates,1183
5,distinct_temporal_buckets,10
6,missing_observation_datetime_rows,0


,weather_station_id,weather_station_name,observation_datetime,date,day_of_week,hour,daypart,temporal_bucket,temporal_bucket_order,observation_sequence_id,pre_post_cp
0,USW00014732,New York LaGuardia Airport,2023-01-01 00:15:00,2023-01-01,Sunday,0,overnight,weekend_overnight,6,1,pre_cp
1,USW00014732,New York LaGuardia Airport,2023-01-01 00:51:00,2023-01-01,Sunday,0,overnight,weekend_overnight,6,1,pre_cp
2,USW00014732,New York LaGuardia Airport,2023-01-01 01:51:00,2023-01-01,Sunday,1,overnight,weekend_overnight,6,1,pre_cp
3,USW00014732,New York LaGuardia Airport,2023-01-01 02:38:00,2023-01-01,Sunday,2,overnight,weekend_overnight,6,1,pre_cp
4,USW00014732,New York LaGuardia Airport,2023-01-01 02:51:00,2023-01-01,Sunday,2,overnight,weekend_overnight,6,1,pre_cp
5,USW00014732,New York LaGuardia Airport,2023-01-01 03:00:00,2023-01-01,Sunday,3,overnight,weekend_overnight,6,1,pre_cp
6,USW00014732,New York LaGuardia Airport,2023-01-01 03:49:00,2023-01-01,Sunday,3,overnight,weekend_overnight,6,1,pre_cp
7,USW00014732,New York LaGuardia Airport,2023-01-01 03:51:00,2023-01-01,Sunday,3,overnight,weekend_overnight,6,1,pre_cp
8,USW00014732,New York LaGuardia Airport,2023-01-01 04:51:00,2023-01-01,Sunday,4,overnight,weekend_overnight,6,1,pre_cp
9,USW00014732,New York LaGuardia Airport,2023-01-01 05:19:00,2023-01-01,Sunday,5,overnight,weekend_overnight,6,1,pre_cp


## 1\.2\.5\.2 Aggregate weather to weather\_station\_id x date x temporal\_bucket

This chapter collapses the raw hourly weather sidecar into something the rest of the project can actually use\. The first step is to stay disciplined about feature scope: keep only the approved weather measures, inspect how they behave in the temporally standardized station\-hour table, and make explicit cleaning decisions before we aggregate anything\. Once those fields are cleaned and locked, we can safely roll weather up to the shared \`weather\_station\_id x date x temporal\_bucket\` grain and validate the resulting outputs against the same lightweight contract style used elsewhere in the mobility foundation\.

### Retain and clean approved weather metrics

This subsection is the gatekeeper for the rest of the weather harmonization workflow\. Before we aggregate anything, we want a compact profile of the approved retained metrics exactly as they arrive from \`1\.2\.5\.1\`: which columns are present, how much missingness they have, whether obvious sentinel or impossible values still remain, and where the main cleaning decisions still need a human judgment call\. The output here should help us separate normal source sparsity from true cleanup work, and it gives us one clear pause point before we codify the cleaning rules\.

In [6]:
# ---------------------------------------------------------------------
# Profile retained weather metric availability
# ---------------------------------------------------------------------

print_step("Loading temporal-standardized weather table for retained-metric profiling.")
weather_temporal_df = pd.read_parquet(WEATHER_TEMPORAL_OUTPUT_PATH)

retained_metric_presence_df = pd.DataFrame(
    {
        "metric": RETAINED_WEATHER_MEASURE_COLUMNS,
        "present": [metric in weather_temporal_df.columns for metric in RETAINED_WEATHER_MEASURE_COLUMNS],
    }
)

missing_retained_metrics = retained_metric_presence_df.loc[
    ~retained_metric_presence_df["present"], "metric"
].tolist()
if missing_retained_metrics:
    raise ValueError(
        f"Temporal weather table is missing retained metrics: {missing_retained_metrics}"
    )

metric_cleaning_specs = {
    "temperature": {
        "lower_bound": -40,
        "upper_bound": 130,
        "sentinel_values": [],
        "notes": "Expected in degrees Fahrenheit from curated 1.1.7 export.",
    },
    "precipitation": {
        "lower_bound": 0,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Event-driven and often zero or null.",
    },
    "visibility": {
        "lower_bound": 0,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Profile first before assuming units in downstream prose.",
    },
    "wind_speed": {
        "lower_bound": 0,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Nonnegative continuous speed field.",
    },
    "wind_gust": {
        "lower_bound": 0,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Expected to be sparse relative to wind_speed.",
    },
    "relative_humidity": {
        "lower_bound": 0,
        "upper_bound": 100,
        "sentinel_values": [],
        "notes": "Values above 100 are likely impossible.",
    },
    "ceiling_height": {
        "lower_bound": 0,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Nonnegative height field.",
    },
    "pressure_3hr_change": {
        "lower_bound": None,
        "upper_bound": None,
        "sentinel_values": [],
        "notes": "Signed change field; review distribution before clipping.",
    },
}

profile_rows = []
for metric, spec in metric_cleaning_specs.items():
    metric_series = weather_temporal_df[metric]
    numeric_series = pd.to_numeric(metric_series, errors="coerce")

    sentinel_count = 0
    if spec["sentinel_values"]:
        sentinel_count = int(metric_series.isin(spec["sentinel_values"]).sum())

    out_of_bounds_mask = pd.Series(False, index=weather_temporal_df.index)
    if spec["lower_bound"] is not None:
        out_of_bounds_mask = out_of_bounds_mask | (numeric_series < spec["lower_bound"])
    if spec["upper_bound"] is not None:
        out_of_bounds_mask = out_of_bounds_mask | (numeric_series > spec["upper_bound"])

    profile_rows.append(
        {
            "metric": metric,
            "row_count": int(len(weather_temporal_df)),
            "non_null_count": int(metric_series.notna().sum()),
            "null_count": int(metric_series.isna().sum()),
            "null_pct": round(float(metric_series.isna().mean() * 100), 4),
            "distinct_non_null_values": int(metric_series.dropna().nunique()),
            "min_value": numeric_series.min(),
            "median_value": numeric_series.median(),
            "max_value": numeric_series.max(),
            "sentinel_value_count": sentinel_count,
            "out_of_bounds_count": int(out_of_bounds_mask.fillna(False).sum()),
            "notes": spec["notes"],
        }
    )

weather_retained_metric_profile_df = pd.DataFrame(profile_rows)
weather_retained_metric_priority_df = weather_retained_metric_profile_df[[
    "metric",
    "null_pct",
    "sentinel_value_count",
    "out_of_bounds_count",
    "notes",
]].copy()
weather_retained_metric_priority_df["recommended_next_step"] = np.select(
    [
        weather_retained_metric_priority_df["out_of_bounds_count"] > 0,
        weather_retained_metric_priority_df["sentinel_value_count"] > 0,
        weather_retained_metric_priority_df["null_pct"] >= 90,
        weather_retained_metric_priority_df["null_pct"] >= 60,
    ],
    [
        "human_review_required",
        "human_review_required",
        "likely_keep_but_sparse",
        "likely_keep_with_missingness_note",
    ],
    default="likely_keep",
)

weather_station_metric_non_null_df = (
    weather_temporal_df.groupby(["weather_station_id", "weather_station_name"], dropna=False)[RETAINED_WEATHER_MEASURE_COLUMNS]
    .agg(lambda s: int(s.notna().sum()))
    .reset_index()
)

print_step("Retained weather metric availability loaded.")
display(retained_metric_presence_df)

[1.2.5] Loading temporal-standardized weather table for retained-metric profiling.
[1.2.5] Retained weather metric availability loaded.


,metric,present
0,temperature,True
1,precipitation,True
2,visibility,True
3,wind_speed,True
4,wind_gust,True
5,relative_humidity,True
6,ceiling_height,True
7,pressure_3hr_change,True


We start with a diagnostic pass only\. This block loads the temporally standardized station\-hour weather table, confirms the retained metric columns are present, and profiles each approved field using the real current values rather than guessed downstream assumptions\. It does not clean or overwrite anything yet; the goal is to tee up the exact keep\-versus\-fix decisions we want to make next\.

In [7]:
# ---------------------------------------------------------------------
# Review retained weather metric profile
# ---------------------------------------------------------------------

weather_retained_metric_review_df = weather_retained_metric_priority_df.merge(
    weather_retained_metric_profile_df[
        [
            "metric",
            "non_null_count",
            "min_value",
            "median_value",
            "max_value",
        ]
    ],
    on="metric",
    how="left",
)[
    [
        "metric",
        "non_null_count",
        "null_pct",
        "min_value",
        "median_value",
        "max_value",
        "out_of_bounds_count",
        "recommended_next_step",
    ]
].copy()

weather_retained_metric_review_df = weather_retained_metric_review_df.sort_values(
    ["recommended_next_step", "metric"]
).reset_index(drop=True)

display(weather_retained_metric_review_df)

,metric,non_null_count,null_pct,min_value,median_value,max_value,out_of_bounds_count,recommended_next_step
0,relative_humidity,160531,27.6689,10.0,64.000,1156.000,1,human_review_required
1,ceiling_height,136930,38.3029,0.0,6096.000,22000.000,0,likely_keep
2,precipitation,103919,53.1768,0.0,0.000,52.800,0,likely_keep
3,temperature,160568,27.6522,-22.0,12.800,39.400,0,likely_keep
4,visibility,160539,27.6653,0.0,16.093,64.374,0,likely_keep
5,wind_speed,157506,29.0319,0.0,3.600,20.600,0,likely_keep
6,pressure_3hr_change,61898,72.1104,-60.0,0.000,8.400,0,likely_keep_with_missingness_note
7,wind_gust,26555,88.0350,7.2,11.300,32.400,0,likely_keep_with_missingness_note


Findings\. The retained weather profile looks pretty workable overall\. Temperature, visibility, wind speed, ceiling height, and precipitation all look usable as\-is for the harmonized path, even if precipitation is naturally sparse and visibility still needs us to stay modest about unit assumptions in the prose\. The only true cleanup flag here is relative humidity, where we have one impossible value above 100 that should be nulled out before aggregation\. Wind gust and 3\-hour pressure change are much sparser than the core weather fields, but not so sparse that they need to be dropped automatically; they read more like secondary contextual features that we should keep with an explicit missingness note rather than treat as uniformly available signals\.

In [8]:
# ---------------------------------------------------------------------
# Review retained metric coverage by station
# ---------------------------------------------------------------------

weather_station_metric_review_df = weather_station_metric_non_null_df.copy()

display(weather_station_metric_review_df)

,weather_station_id,weather_station_name,temperature,precipitation,visibility,wind_speed,wind_gust,relative_humidity,ceiling_height,pressure_3hr_change
0,USW00014732,New York LaGuardia Airport,42423,26218,42423,42410,7554,42423,34473,17984
1,USW00014734,Newark Liberty International Airport,40343,25584,40350,40319,7023,40343,33193,16690
2,USW00094728,NY CITY CNTRL PARK,36305,27471,36261,33280,6143,36268,35603,9296
3,USW00094789,New York JFK Airport,41497,24646,41505,41497,5835,41497,33661,17928


Findings\. Station coverage is also reassuringly balanced across the four retained stations\. LaGuardia, Newark, and JFK are all very close to one another on the core fields, while Central Park is a bit lighter on wind speed, wind gust, and especially pressure\-change coverage, but still has strong support for temperature, visibility, humidity, precipitation, and ceiling height\. So this does not look like one station is fundamentally broken; it looks more like normal station\-level reporting differences, which means we can keep the shared retained metric set and handle sparsity through cleaning notes and downstream aggregation rather than by carving out station\-specific rules\.

> Decision\. We will keep the curated weather metrics in the harmonized raw layer, null out the one impossible relative\-humidity value above 100, and otherwise preserve source missingness rather than filling it here\. Temperature, precipitation, visibility, wind speed, and ceiling height can move forward as\-is; wind gust and 3\-hour pressure change stay in scope as sparser secondary context fields, with the understanding that their missingness is a source characteristic rather than something we should fix inside the canonical weather table\.

Decision\. We are not making the Taxi Zone weather\-assignment choice yet in this chapter, but we are also not punting it indefinitely\. The station\-to\-zone question will be tested explicitly in \`1\.2\.5\.3\`, where we can compare nearest\-station assignment against the possibility that Central Park reflects a meaningfully different local weather regime\. That keeps \`1\.2\.5\.2\` focused on producing a clean station\-level temporal weather layer, while \`1\.2\.5\.3\` becomes the right place to decide whether zone\-level assignment is actually defensible\.

This block applies the one cleaning rule we already have enough evidence to justify\. Rather than clipping values or inventing replacements, we keep the retained station\-hour weather table intact and only convert impossible relative\-humidity readings above 100 to null\. The goal is to remove clear invalidity while preserving the original missingness pattern for later EDA and task\-specific model exports\.

In [9]:
# ---------------------------------------------------------------------
# Clean retained weather metrics
# ---------------------------------------------------------------------

weather_cleaned_df = weather_temporal_df.copy()

invalid_relative_humidity_mask = weather_cleaned_df["relative_humidity"] > 100
invalid_relative_humidity_rows = int(invalid_relative_humidity_mask.fillna(False).sum())

weather_cleaned_df.loc[
    invalid_relative_humidity_mask,
    "relative_humidity",
] = np.nan

weather_cleaning_summary_df = pd.DataFrame(
    {
        "metric": ["relative_humidity"],
        "rule_applied": ["set values > 100 to null"],
        "rows_affected": [invalid_relative_humidity_rows],
        "remaining_values_above_100": [
            int((weather_cleaned_df["relative_humidity"] > 100).fillna(False).sum())
        ],
    }
)

print_step("Retained weather cleaning step complete.")
display(weather_cleaning_summary_df)

[1.2.5] Retained weather cleaning step complete.


,metric,rule_applied,rows_affected,remaining_values_above_100
0,relative_humidity,set values > 100 to null,1,0


Findings\. The cleaning step was exactly as light\-touch as intended\. Only one relative\-humidity record needed intervention, and after converting values above 100 to null, no impossible humidity values remain\. That gives us a cleaner weather layer without distorting the raw signal or introducing any made\-up replacements\.

### Diagnose station\-hour reporting density before bucket aggregation

This subsection tests whether the cleaned weather table is actually ready to be rolled up to the station\-date\-temporal\-bucket grain\. Rather than assuming each row already represents one clean hourly observation, we inspect within\-hour reporting density, repeated station\-hour groups, and the structure of the worst offending cases\. The goal is to determine whether direct bucket aggregation is defensible or whether we first need to standardize the weather data to one record per weather\_station\_id x observation\_hour\.

This first aggregation block is still diagnostic\. It summarizes how many hourly observations are available inside each \`weather\_station\_id x date x temporal\_bucket\` group, shows how often buckets are fully populated versus partial, and previews metric\-level completeness at the bucket grain\. That gives us the evidence we need to decide whether simple means are good enough for the continuous fields and whether precipitation should remain a sum\-like field at aggregation time\.

In [10]:
# ---------------------------------------------------------------------
# Profile station-date-temporal-bucket aggregation readiness
# ---------------------------------------------------------------------

bucket_key_columns = [
    "weather_station_id",
    "weather_station_name",
    "date",
    "day_of_week",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
]

weather_bucket_hourly_density_df = (
    weather_cleaned_df.groupby(bucket_key_columns, dropna=False)
    .agg(
        hourly_observation_count=("observation_datetime", "size"),
        min_hour=("hour", "min"),
        max_hour=("hour", "max"),
    )
    .reset_index()
)

weather_bucket_density_summary_df = pd.DataFrame(
    {
        "metric": [
            "bucket_row_count",
            "station_count",
            "min_hourly_observation_count",
            "median_hourly_observation_count",
            "max_hourly_observation_count",
            "single_hour_buckets",
            "buckets_with_2plus_hours",
            "buckets_with_4plus_hours",
        ],
        "value": [
            int(len(weather_bucket_hourly_density_df)),
            int(weather_bucket_hourly_density_df["weather_station_id"].nunique()),
            int(weather_bucket_hourly_density_df["hourly_observation_count"].min()),
            float(weather_bucket_hourly_density_df["hourly_observation_count"].median()),
            int(weather_bucket_hourly_density_df["hourly_observation_count"].max()),
            int((weather_bucket_hourly_density_df["hourly_observation_count"] == 1).sum()),
            int((weather_bucket_hourly_density_df["hourly_observation_count"] >= 2).sum()),
            int((weather_bucket_hourly_density_df["hourly_observation_count"] >= 4).sum()),
        ],
    }
)

bucket_metric_non_null_rows = []
for metric in RETAINED_WEATHER_MEASURE_COLUMNS:
    metric_bucket_summary = (
        weather_cleaned_df.groupby(bucket_key_columns, dropna=False)[metric]
        .agg(non_null_hour_count=lambda s: int(s.notna().sum()))
        .reset_index()
    )
    bucket_metric_non_null_rows.append(
        {
            "metric": metric,
            "bucket_rows": int(len(metric_bucket_summary)),
            "buckets_with_any_non_null": int((metric_bucket_summary["non_null_hour_count"] > 0).sum()),
            "buckets_with_2plus_non_null": int((metric_bucket_summary["non_null_hour_count"] >= 2).sum()),
            "median_non_null_hours_per_bucket": float(metric_bucket_summary["non_null_hour_count"].median()),
            "max_non_null_hours_per_bucket": int(metric_bucket_summary["non_null_hour_count"].max()),
        }
    )

weather_bucket_metric_readiness_df = pd.DataFrame(bucket_metric_non_null_rows)

bucket_density_distribution_df = (
    weather_bucket_hourly_density_df["hourly_observation_count"]
    .value_counts()
    .sort_index()
    .reset_index()
)
bucket_density_distribution_df.columns = ["hourly_observation_count", "bucket_count"]

print_step("Station-date-temporal-bucket aggregation readiness profile complete.")
display(weather_bucket_density_summary_df)

[1.2.5] Station-date-temporal-bucket aggregation readiness profile complete.


,metric,value
0,bucket_row_count,23648.0
1,station_count,4.0
2,min_hourly_observation_count,1.0
3,median_hourly_observation_count,6.0
4,max_hourly_observation_count,91.0
5,single_hour_buckets,4.0
6,buckets_with_2plus_hours,23644.0
7,buckets_with_4plus_hours,23578.0


Findings\. The bucket structure looks strong overall\. We end up with 23,648 station\-date\-temporal\-bucket rows across all four stations, and the typical bucket has six hourly observations, which is a pretty healthy base for within\-bucket summaries\. Almost every bucket has at least two observations, and nearly all have four or more, so the station\-bucket grain looks well supported rather than flimsy\. The one thing that stands out is the very high maximum of 91 hourly observations in a single bucket, which suggests we may still have some duplicate\-within\-hour or multiple\-report\-per\-hour behavior that is worth understanding before we finalize the aggregation rule\.

In [11]:
# ---------------------------------------------------------------------
# Review metric readiness at the bucket grain
# ---------------------------------------------------------------------

display(weather_bucket_metric_readiness_df)

,metric,bucket_rows,buckets_with_any_non_null,buckets_with_2plus_non_null,median_non_null_hours_per_bucket,max_non_null_hours_per_bucket
0,temperature,23648,23640,23632,6.0,33
1,precipitation,23648,19882,19721,4.0,31
2,visibility,23648,23634,23624,6.0,33
3,wind_speed,23648,23619,23562,6.0,31
4,wind_gust,23648,9231,6351,0.0,20
5,relative_humidity,23648,23638,23629,6.0,33
6,ceiling_height,23648,23573,23217,5.0,33
7,pressure_3hr_change,23648,23512,20719,3.0,5


Findings\. Metric readiness at the bucket grain is also encouraging\. Temperature, visibility, wind speed, relative humidity, and ceiling height are available in nearly every bucket and usually have multiple non\-null hourly observations, which makes them good candidates for straightforward bucket summaries\. Precipitation is less complete but still broadly usable, while wind gust is clearly the sparsest field by far and pressure change is meaningfully thinner than the core measures\. That does not force us to drop either one, but it does reinforce the idea that they should be treated as secondary contextual features rather than relied on as uniformly dense inputs\.

In [12]:
# ---------------------------------------------------------------------
# Review hourly density distribution within buckets
# ---------------------------------------------------------------------

display(bucket_density_distribution_df)

,hourly_observation_count,bucket_count
0,1,4
1,2,18
2,3,48
3,4,2813
4,5,5118
...,...,...
85,86,4
86,87,4
87,89,1
88,90,1


Findings\. The hourly\-density distribution shows that most station\-date\-temporal\-bucket groups are built from a sensible multi\-hour base rather than a single isolated reading\. Buckets are concentrated most heavily in the four\-to\-eight observation range, with especially large counts at five, six, and eight hourly observations, which is broadly consistent with the expected width of the project dayparts\. Only a tiny number of buckets have just one to three observations, so thin bucket coverage looks like the exception rather than the rule\.

Before we lock the bucket aggregation rules, we need to understand whether unusually large within\-bucket counts are being caused by multiple weather observations inside the same station\-hour\. This diagnostic sequence checks exactly that: how many repeated station\-hour observations exist, whether they are concentrated in particular stations or buckets, and what the worst offending groups actually look like\. If the issue is real, the likely implication is that we should collapse to one record per \`weather\_station\_id x observation\_hour\` before rolling up to temporal buckets\.

In [13]:
# ---------------------------------------------------------------------
# Diagnose repeated observations within station-hour groups
# ---------------------------------------------------------------------

station_hour_key_columns = [
    "weather_station_id",
    "weather_station_name",
    "date",
    "observation_hour",
    "day_of_week",
    "daypart",
    "temporal_bucket",
    "observation_sequence_id",
]

weather_station_hour_density_df = (
    weather_cleaned_df.groupby(station_hour_key_columns, dropna=False)
    .agg(
        raw_observation_count=("observation_datetime", "size"),
        min_observation_datetime=("observation_datetime", "min"),
        max_observation_datetime=("observation_datetime", "max"),
    )
    .reset_index()
)

weather_station_hour_repeat_summary_df = pd.DataFrame(
    {
        "metric": [
            "station_hour_rows",
            "station_hour_rows_with_single_observation",
            "station_hour_rows_with_repeats",
            "max_raw_observations_per_station_hour",
            "median_raw_observations_per_station_hour",
        ],
        "value": [
            int(len(weather_station_hour_density_df)),
            int((weather_station_hour_density_df["raw_observation_count"] == 1).sum()),
            int((weather_station_hour_density_df["raw_observation_count"] > 1).sum()),
            int(weather_station_hour_density_df["raw_observation_count"].max()),
            float(weather_station_hour_density_df["raw_observation_count"].median()),
        ],
    }
)

weather_station_hour_repeat_groups_df = weather_station_hour_density_df.loc[
    weather_station_hour_density_df["raw_observation_count"] > 1
].copy()

print_step("Station-hour repeat diagnostic summary ready.")
display(weather_station_hour_repeat_summary_df)

[1.2.5] Station-hour repeat diagnostic summary ready.


,metric,value
0,station_hour_rows,113325.0
1,station_hour_rows_with_single_observation,72070.0
2,station_hour_rows_with_repeats,41255.0
3,max_raw_observations_per_station_hour,18.0
4,median_raw_observations_per_station_hour,1.0


Findings\. Repeated observations within station\-hour groups are common enough that we should take them seriously before finalizing the bucket aggregation logic\. Out of 113,325 total station\-hour rows, 41,255 have more than one raw observation, even though the median station\-hour still contains just one record\. The fact that the worst station\-hour reaches 18 observations confirms that the oversized bucket counts are not random noise; they are being driven by multiple intra\-hour reports flowing into the same station\-hour and then into the same temporal bucket\.

This block rolls the repeated station\-hour issue up to the station level so we can see whether the behavior is broadly shared or concentrated in a smaller subset of stations\. The goal is to compare how often repeated station\-hour groups occur across stations, how severe those repeats get, and whether one station appears disproportionately responsible for the inflated within\-bucket observation counts\.

In [14]:
# ---------------------------------------------------------------------
# Review repeated station-hour observations by station
# ---------------------------------------------------------------------

weather_station_hour_repeat_by_station_df = (
    weather_station_hour_repeat_groups_df.groupby(
        ["weather_station_id", "weather_station_name"],
        dropna=False,
    )
    .agg(
        repeated_station_hour_rows=("raw_observation_count", "size"),
        max_raw_observations_per_station_hour=("raw_observation_count", "max"),
        median_raw_observations_per_station_hour=("raw_observation_count", "median"),
    )
    .reset_index()
    .sort_values(
        ["repeated_station_hour_rows", "max_raw_observations_per_station_hour"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

display(weather_station_hour_repeat_by_station_df)

,weather_station_id,weather_station_name,repeated_station_hour_rows,max_raw_observations_per_station_hour,median_raw_observations_per_station_hour
0,USW00014732,New York LaGuardia Airport,12351,18,2.0
1,USW00094789,New York JFK Airport,11812,15,2.0
2,USW00014734,Newark Liberty International Airport,11342,16,2.0
3,USW00094728,NY CITY CNTRL PARK,5750,18,3.0


Findings\. The worst offending groups confirm that this is a real intra\-hour reporting pattern rather than a keying mistake\. We see single station\-hour groups with 16 to 18 raw observations, all packed within the same hour window, and the issue appears across multiple stations rather than just one broken source\. Central Park and LaGuardia show up especially often in the top examples, but Newark appears too, which suggests we are looking at a broader source\-reporting behavior\. That means direct bucket aggregation from the current raw timestamps would overweight hours that happened to be reported more frequently\.

This block isolates the most extreme repeated station\-hour groups so we can see what is driving the inflated within\-bucket observation counts\. The goal is to surface the worst cases, check whether the repeats stay confined to single hours, and see whether the pattern is concentrated in certain stations, dates, or temporal buckets before we decide how to standardize the hourly weather layer\.

In [15]:
# ---------------------------------------------------------------------
# Review worst offending repeated station-hour groups
# ---------------------------------------------------------------------

weather_station_hour_repeat_examples_df = weather_station_hour_repeat_groups_df.sort_values(
    ["raw_observation_count", "weather_station_id", "observation_hour"],
    ascending=[False, True, True],
).head(20).reset_index(drop=True)

display(
    weather_station_hour_repeat_examples_df[
        [
            "weather_station_id",
            "weather_station_name",
            "date",
            "observation_hour",
            "temporal_bucket",
            "raw_observation_count",
            "min_observation_datetime",
            "max_observation_datetime",
        ]
    ]
)

,weather_station_id,weather_station_name,date,observation_hour,temporal_bucket,raw_observation_count,min_observation_datetime,max_observation_datetime
0,USW00014732,New York LaGuardia Airport,2026-03-17,2026-03-17 04:00:00,weekday_overnight,18,2026-03-17 04:00:00,2026-03-17 04:55:00
1,USW00094728,NY CITY CNTRL PARK,2026-01-01,2026-01-01 11:00:00,weekday_midday,18,2026-01-01 11:00:00,2026-01-01 11:55:00
2,USW00014732,New York LaGuardia Airport,2026-01-18,2026-01-18 12:00:00,weekend_midday,17,2026-01-18 12:00:00,2026-01-18 12:58:00
3,USW00014732,New York LaGuardia Airport,2026-03-12,2026-03-12 05:00:00,weekday_overnight,17,2026-03-12 05:00:00,2026-03-12 05:55:00
4,USW00094728,NY CITY CNTRL PARK,2026-01-19,2026-01-19 02:00:00,weekday_overnight,17,2026-01-19 02:00:00,2026-01-19 02:55:00
5,USW00094728,NY CITY CNTRL PARK,2026-01-26,2026-01-26 09:00:00,weekday_am_peak,17,2026-01-26 09:00:00,2026-01-26 09:55:00
6,USW00094728,NY CITY CNTRL PARK,2026-02-16,2026-02-16 07:00:00,weekday_am_peak,17,2026-02-16 07:00:00,2026-02-16 07:57:00
7,USW00094728,NY CITY CNTRL PARK,2026-02-18,2026-02-18 15:00:00,weekday_midday,17,2026-02-18 15:00:00,2026-02-18 15:58:00
8,USW00094728,NY CITY CNTRL PARK,2026-03-05,2026-03-05 14:00:00,weekday_midday,17,2026-03-05 14:00:00,2026-03-05 14:55:00
9,USW00094728,NY CITY CNTRL PARK,2026-03-07,2026-03-07 12:00:00,weekend_midday,17,2026-03-07 12:00:00,2026-03-07 12:55:00


Findings\. The worst offending groups make the issue concrete: we are seeing 16 to 18 raw observations packed inside single station\-hour windows, all within the same hour span from :00 to roughly :55 or :59\. The pattern is not isolated to one bad row or one broken station either; LaGuardia, Central Park, and Newark all appear among the top repeated groups, with especially frequent heavy repeats at LaGuardia and Central Park\. That strongly suggests a real source\-reporting pattern of multiple subhour weather observations, which means direct temporal\-bucket aggregation from the current raw timestamps would overweight hours that happened to be reported more often\.

This final spot check drills into one of the worst repeated station\-hour groups to show exactly what the raw timestamps look like inside the hour\. The point is to verify whether the repeated rows are true duplicates, partial within\-hour reports, or a sequence of distinct subhour observations carrying different field completeness\. That distinction matters because it determines whether we should collapse to one station\-hour record before any bucket\-level aggregation\.

In [16]:
# ---------------------------------------------------------------------
# Review raw timestamps for the single worst station-hour group
# ---------------------------------------------------------------------

worst_station_hour_group_df = weather_station_hour_repeat_examples_df.head(1).copy()

if worst_station_hour_group_df.empty:
    weather_station_hour_raw_trace_df = pd.DataFrame()
else:
    worst_station_id = worst_station_hour_group_df.iloc[0]["weather_station_id"]
    worst_observation_hour = worst_station_hour_group_df.iloc[0]["observation_hour"]

    weather_station_hour_raw_trace_df = (
        weather_cleaned_df[
            (weather_cleaned_df["weather_station_id"] == worst_station_id)
            & (weather_cleaned_df["observation_hour"] == worst_observation_hour)
        ]
        .sort_values("observation_datetime")
        .reset_index(drop=True)
    )

display(
    weather_station_hour_raw_trace_df[
        [
            "weather_station_id",
            "weather_station_name",
            "observation_datetime",
            "observation_hour",
            "temperature",
            "precipitation",
            "visibility",
            "wind_speed",
            "relative_humidity",
        ]
    ]
)

,weather_station_id,weather_station_name,observation_datetime,observation_hour,temperature,precipitation,visibility,wind_speed,relative_humidity
0,USW00014732,New York LaGuardia Airport,2026-03-17 04:00:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
1,USW00014732,New York LaGuardia Airport,2026-03-17 04:05:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
2,USW00014732,New York LaGuardia Airport,2026-03-17 04:07:00,2026-03-17 04:00:00,11.1,NaN,0.805,11.8,97.0
3,USW00014732,New York LaGuardia Airport,2026-03-17 04:10:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
4,USW00014732,New York LaGuardia Airport,2026-03-17 04:15:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
5,USW00014732,New York LaGuardia Airport,2026-03-17 04:20:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
6,USW00014732,New York LaGuardia Airport,2026-03-17 04:21:00,2026-03-17 04:00:00,11.7,NaN,3.219,14.4,96.0
7,USW00014732,New York LaGuardia Airport,2026-03-17 04:25:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
8,USW00014732,New York LaGuardia Airport,2026-03-17 04:30:00,2026-03-17 04:00:00,NaN,NaN,NaN,NaN,NaN
9,USW00014732,New York LaGuardia Airport,2026-03-17 04:33:00,2026-03-17 04:00:00,11.7,0.0,3.219,13.4,93.0


Findings\. This trace makes the pattern very clear: the repeated rows are not exact duplicates, but a sequence of subhour observations within the same 4:00 AM hour, many of which are sparse shell records with all key weather fields missing, interspersed with a smaller number of more complete observations\. In other words, the source is emitting multiple partial within\-hour reports, not one clean hourly reading\. That strongly argues for adding a weather\_station\_id x observation\_hour consolidation step before temporal\-bucket aggregation, so that each hour contributes once to the bucket instead of being overweighted by reporting frequency\.

> ⚖️ Decision\. We will not aggregate directly from raw weather timestamps to the weather\_station\_id x date x temporal\_bucket grain\. The diagnostic review showed that many station\-hour groups contain multiple subhour observations, including sparse shell records and a smaller number of fuller reports, which means direct bucket aggregation would overweight hours that happened to be reported more frequently\. Instead, we will introduce an intermediate weather\_station\_id x observation\_hour consolidation step, then aggregate those standardized station\-hour records up to the station\-date\-bucket level\. At the metric level, temperature, visibility, wind speed, relative humidity, ceiling height, and pressure\-change fields will use hourly means rolled into bucket means; precipitation will use hourly sums rolled into bucket sums; and wind gust will use hourly maxima rolled into bucket maxima\.

### Build canonical station\-hour weather observations

This subsection creates the clean intermediate weather grain that the earlier diagnostics showed we need\. We will collapse repeated subhour observations into one canonical weather\_station\_id x observation\_hour record using metric\-specific hourly rules, preserve the shared temporal fields needed downstream, and validate that the resulting station\-hour table is unique and well\-formed before using it as the input to station\-date\-temporal\-bucket aggregation\.

In [17]:
# ---------------------------------------------------------------------
# Build canonical station-hour weather observations
# ---------------------------------------------------------------------

station_hour_group_columns = [
    "weather_station_id",
    "weather_station_name",
    "observation_hour",
    "date",
    "year",
    "month",
    "day_of_week",
    "hour",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
]

if should_build(
    WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH,
    REBUILD_WEATHER_CANONICAL_STATION_HOUR,
):
    weather_station_hour_df = (
        weather_cleaned_df.groupby(station_hour_group_columns, dropna=False)
        .agg(
            raw_observation_count=("observation_datetime", "size"),
            temperature=("temperature", "mean"),
            precipitation=("precipitation", "sum"),
            visibility=("visibility", "mean"),
            wind_speed=("wind_speed", "mean"),
            wind_gust=("wind_gust", "max"),
            relative_humidity=("relative_humidity", "mean"),
            ceiling_height=("ceiling_height", "mean"),
            pressure_3hr_change=("pressure_3hr_change", "mean"),
        )
        .reset_index()
    )

    weather_station_hour_df["temporal_grain_type"] = "canonical_station_hour"
    weather_station_hour_df["date_assignment_method"] = "derived_from_native_station_timestamp"

    weather_station_hour_df.to_parquet(
        WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH,
        index=False,
    )
    print_step(
        f"Wrote canonical station-hour weather output: {WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH}"
    )
else:
    print_step("Loading existing canonical station-hour weather output.")
    weather_station_hour_df = pd.read_parquet(
        WEATHER_CANONICAL_STATION_HOUR_OUTPUT_PATH
    )

weather_station_hour_summary_df = pd.DataFrame(
    {
        "metric": [
            "canonical_station_hour_rows",
            "station_count",
            "min_observation_hour",
            "max_observation_hour",
            "max_raw_observation_count_collapsed",
        ],
        "value": [
            int(len(weather_station_hour_df)),
            int(weather_station_hour_df["weather_station_id"].nunique()),
            weather_station_hour_df["observation_hour"].min(),
            weather_station_hour_df["observation_hour"].max(),
            int(weather_station_hour_df["raw_observation_count"].max()),
        ],
    }
)

display(weather_station_hour_summary_df)

[1.2.5] Loading existing canonical station-hour weather output.


,metric,value
0,canonical_station_hour_rows,113325
1,station_count,4
2,min_observation_hour,2023-01-01 00:00:00
3,max_observation_hour,2026-03-31 23:00:00
4,max_raw_observation_count_collapsed,18


Findings\. The canonical station\-hour build did exactly what we wanted it to do: it produced a clean hourly intermediate layer with 113,325 station\-hour rows across four stations, spanning the full study window from January 1, 2023 through March 31, 2026\. Just as importantly, the build preserves visibility into the source reporting density, with some canonical hours still reflecting as many as 18 raw subhour observations collapsed into a single hour\-level record\. That means we have standardized the grain without losing the ability to audit how much within\-hour consolidation actually occurred\.

With the canonical station\-hour table built, the next question is whether it behaves the way we intended\. This review block checks the structural contract at the new grain: one row per station\-hour, preserved study\-period coverage, and a compact view of how much raw reporting density is now being absorbed into each canonical hourly record\.

In [18]:
# ---------------------------------------------------------------------
# Review canonical station-hour structural validity
# ---------------------------------------------------------------------

station_hour_duplicate_rows = int(
    weather_station_hour_df.duplicated(
        subset=["weather_station_id", "observation_hour"],
        keep=False,
    ).sum()
)

weather_station_hour_validation_df = pd.DataFrame(
    {
        "check": [
            "unique_station_hour_key",
            "non_null_station_id",
            "non_null_observation_hour",
            "study_period_hour_bounds",
            "positive_row_count",
        ],
        "status": [
            station_hour_duplicate_rows == 0,
            bool(weather_station_hour_df["weather_station_id"].notna().all()),
            bool(weather_station_hour_df["observation_hour"].notna().all()),
            bool(weather_station_hour_df["observation_hour"].min() >= STUDY_START_DATE)
            and bool(weather_station_hour_df["observation_hour"].max() < (STUDY_END_DATE + pd.Timedelta(days=1))),
            bool(len(weather_station_hour_df) > 0),
        ],
    }
)

display(weather_station_hour_validation_df)

,check,status
0,unique_station_hour_key,True
1,non_null_station_id,True
2,non_null_observation_hour,True
3,study_period_hour_bounds,True
4,positive_row_count,True


Findings\. The structural validation passed cleanly across the board\. The new table has a unique weather\_station\_id x observation\_hour key, no missing station IDs or observation hours, valid study\-period bounds, and a positive row count\. In other words, the canonical station\-hour layer is behaving like a proper intermediate contract rather than just a convenience transform\.

This block shows what the station\-hour consolidation step actually did for us\. We want to see how many raw subhour weather rows got absorbed into the canonical hour\-level table, how often we had to collapse multiple reports into one station\-hour record, and whether this step really fixed the reporting\-density problem we found earlier\.

In [19]:
# ---------------------------------------------------------------------
# Review canonical station-hour reporting collapse
# ---------------------------------------------------------------------

weather_station_hour_collapse_df = pd.DataFrame(
    {
        "metric": [
            "raw_weather_rows",
            "canonical_station_hour_rows",
            "collapsed_rows_removed",
            "station_hour_rows_with_multi_report_collapse",
            "max_raw_observation_count_per_station_hour",
        ],
        "value": [
            int(len(weather_cleaned_df)),
            int(len(weather_station_hour_df)),
            int(len(weather_cleaned_df) - len(weather_station_hour_df)),
            int((weather_station_hour_df["raw_observation_count"] > 1).sum()),
            int(weather_station_hour_df["raw_observation_count"].max()),
        ],
    }
)

display(weather_station_hour_collapse_df)

,metric,value
0,raw_weather_rows,221939
1,canonical_station_hour_rows,113325
2,collapsed_rows_removed,108614
3,station_hour_rows_with_multi_report_collapse,41255
4,max_raw_observation_count_per_station_hour,18


Findings\. The reporting\-collapse summary confirms that this step was necessary and meaningful\. We reduced 221,939 raw weather rows down to 113,325 canonical station\-hour rows, removing 108,614 excess subhour rows while still retaining 41,255 station\-hour groups that had required multi\-report consolidation\. That is a substantial normalization of reporting density, and it directly solves the overweighting problem we found in the earlier bucket diagnostics\.

This block checks how the retained weather fields look after we collapse the subhour reports down to one row per station\-hour\. We want to confirm that the core weather measures still look strong at this new grain, while also seeing which secondary fields stay sparse before we move on to the next aggregation step\.

In [20]:
# ---------------------------------------------------------------------
# Review retained metric completeness at the canonical station-hour grain
# ---------------------------------------------------------------------

canonical_station_hour_metric_readiness_df = pd.DataFrame(
    [
        {
            "metric": metric,
            "non_null_station_hour_rows": int(weather_station_hour_df[metric].notna().sum()),
            "null_pct": round(float(weather_station_hour_df[metric].isna().mean() * 100), 4),
        }
        for metric in RETAINED_WEATHER_MEASURE_COLUMNS
    ]
)

display(canonical_station_hour_metric_readiness_df)

,metric,non_null_station_hour_rows,null_pct
0,temperature,113223,0.0900
1,precipitation,113325,0.0000
2,visibility,113169,0.1377
3,wind_speed,110958,2.0887
4,wind_gust,23554,79.2155
5,relative_humidity,113202,0.1085
6,ceiling_height,108902,3.9029
7,pressure_3hr_change,61894,45.3836


Findings\. Metric completeness at the canonical station\-hour grain looks strong for the core weather fields\. Temperature, visibility, relative humidity, and precipitation are now effectively complete at the hour level, while wind speed and ceiling height remain very usable with only light residual missingness\. Wind gust stays highly sparse and pressure change remains materially thinner than the rest, which is consistent with what we already learned upstream: both are still worth carrying forward, but they should be treated as secondary contextual fields rather than assumed to be uniformly populated like the core hourly weather measures\.

> ⚖️ Decision\. We have enough evidence to move forward without any additional investigation at this stage\. The canonical station\-hour layer solved the reporting\-density problem we found upstream, validated cleanly as a unique weather\_station\_id x observation\_hour table, and preserved strong usability for the core weather fields\. The hourly collapse rules therefore look faithful enough to the source behavior to serve as the standard intermediate grain for the next aggregation step\.

### Validate station\-temporal weather outputs

This last subsection checks whether the finished station\-date\-temporal\-bucket weather table is solid enough to hand off to the spatial chapter\. We want to confirm the grain, keys, study\-period coverage, and retained metric population all look right, so \`1\.2\.5\.3\` can focus on spatial assignment rather than cleaning up leftover temporal issues\.

Now that the canonical station\-hour layer is locked, we can do the actual rollup to the station\-date\-temporal\-bucket grain\. This block builds the final station\-bucket weather table from the hour\-level intermediate, using the same metric rules we already approved, so the validation section below has a real output to check instead of a missing file\.

In [21]:
# ---------------------------------------------------------------------
# Build station-date-temporal-bucket weather output
# ---------------------------------------------------------------------

station_bucket_group_columns = [
    "weather_station_id",
    "weather_station_name",
    "date",
    "year",
    "month",
    "day_of_week",
    "daypart",
    "daypart_order",
    "temporal_bucket",
    "temporal_bucket_order",
    "observation_sequence_id",
    "pre_post_cp",
]

if should_build(
    WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH,
    REBUILD_WEATHER_STATION_BUCKET,
):
    weather_station_temporal_bucket_df = (
        weather_station_hour_df.groupby(station_bucket_group_columns, dropna=False)
        .agg(
            station_hour_count=("observation_hour", "size"),
            temperature=("temperature", "mean"),
            precipitation=("precipitation", "sum"),
            visibility=("visibility", "mean"),
            wind_speed=("wind_speed", "mean"),
            wind_gust=("wind_gust", "max"),
            relative_humidity=("relative_humidity", "mean"),
            ceiling_height=("ceiling_height", "mean"),
            pressure_3hr_change=("pressure_3hr_change", "mean"),
        )
        .reset_index()
    )

    weather_station_temporal_bucket_df["temporal_grain_type"] = "canonical_station_date_temporal_bucket"
    weather_station_temporal_bucket_df["date_assignment_method"] = "derived_from_canonical_station_hour"

    weather_station_temporal_bucket_df.to_parquet(
        WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH,
        index=False,
    )
    print_step(
        f"Wrote station-bucket weather output: {WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH}"
    )
else:
    print_step("Loading existing station-bucket weather output.")
    weather_station_temporal_bucket_df = pd.read_parquet(
        WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH
    )

weather_station_bucket_build_summary_df = pd.DataFrame(
    {
        "metric": [
            "station_bucket_rows",
            "station_count",
            "min_date",
            "max_date",
            "distinct_temporal_buckets",
            "max_station_hour_count_per_bucket",
        ],
        "value": [
            int(len(weather_station_temporal_bucket_df)),
            int(weather_station_temporal_bucket_df["weather_station_id"].nunique()),
            weather_station_temporal_bucket_df["date"].min(),
            weather_station_temporal_bucket_df["date"].max(),
            int(weather_station_temporal_bucket_df["temporal_bucket"].nunique()),
            int(weather_station_temporal_bucket_df["station_hour_count"].max()),
        ],
    }
)

display(weather_station_bucket_build_summary_df)

[1.2.5] Loading existing station-bucket weather output.


,metric,value
0,station_bucket_rows,23648
1,station_count,4
2,min_date,2023-01-01 00:00:00
3,max_date,2026-03-31 00:00:00
4,distinct_temporal_buckets,10
5,max_station_hour_count_per_bucket,6


Findings\. The final station\-bucket build looks clean and complete\. We ended up with 23,648 weather\_station\_id x date x temporal\_bucket rows across all four stations, covering the full study window from January 1, 2023 through March 31, 2026 and preserving all 10 project temporal buckets\. Just as importantly, the maximum station\-hour count per bucket is now 6, which is exactly the kind of bounded within\-bucket density we wanted after introducing the canonical station\-hour step\.

This block does the top\-level contract check on the station\-bucket output\. We want one row per \`weather\_station\_id x date x temporal\_bucket\`, clean study\-period bounds, and no missing core key fields before we let this table move downstream\.

In [22]:
# ---------------------------------------------------------------------
# Validate station-bucket structural contract
# ---------------------------------------------------------------------

if "weather_station_temporal_bucket_df" not in globals():
    if WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH.exists():
        weather_station_temporal_bucket_df = pd.read_parquet(
            WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH
        )
    else:
        raise FileNotFoundError(
            "Station-bucket weather output does not exist yet. Build it before running validation."
        )

station_bucket_duplicate_rows = int(
    weather_station_temporal_bucket_df.duplicated(
        subset=["weather_station_id", "date", "temporal_bucket"],
        keep=False,
    ).sum()
)

weather_station_bucket_validation_df = pd.DataFrame(
    {
        "check": [
            "unique_station_date_bucket_key",
            "non_null_station_id",
            "non_null_date",
            "non_null_temporal_bucket",
            "study_period_date_bounds",
            "positive_row_count",
        ],
        "status": [
            station_bucket_duplicate_rows == 0,
            bool(weather_station_temporal_bucket_df["weather_station_id"].notna().all()),
            bool(weather_station_temporal_bucket_df["date"].notna().all()),
            bool(weather_station_temporal_bucket_df["temporal_bucket"].notna().all()),
            bool(weather_station_temporal_bucket_df["date"].min() >= STUDY_START_DATE)
            and bool(weather_station_temporal_bucket_df["date"].max() <= STUDY_END_DATE),
            bool(len(weather_station_temporal_bucket_df) > 0),
        ],
    }
)

display(weather_station_bucket_validation_df)

,check,status
0,unique_station_date_bucket_key,True
1,non_null_station_id,True
2,non_null_date,True
3,non_null_temporal_bucket,True
4,study_period_date_bounds,True
5,positive_row_count,True


Findings\. The structural contract passed cleanly across every check\. The station\-bucket table has a unique weather\_station\_id x date x temporal\_bucket key, no missing station IDs, dates, or temporal buckets, valid study\-period bounds, and a positive row count\. In other words, the temporal weather layer is now in a stable enough shape to hand off downstream without carrying unresolved grain problems with it\.

This block checks whether the retained weather fields still look usable after the hour\-to\-bucket rollup\. We mainly want to see whether the core fields stay broadly populated and whether the sparser fields still look like context rather than dead weight\.

In [23]:
# ---------------------------------------------------------------------
# Review retained metric completeness at the station-bucket grain
# ---------------------------------------------------------------------

station_bucket_metric_readiness_df = pd.DataFrame(
    [
        {
            "metric": metric,
            "non_null_station_bucket_rows": int(weather_station_temporal_bucket_df[metric].notna().sum()),
            "null_pct": round(float(weather_station_temporal_bucket_df[metric].isna().mean() * 100), 4),
        }
        for metric in RETAINED_WEATHER_MEASURE_COLUMNS
    ]
)

display(station_bucket_metric_readiness_df)

,metric,non_null_station_bucket_rows,null_pct
0,temperature,23640,0.0338
1,precipitation,23648,0.0000
2,visibility,23634,0.0592
3,wind_speed,23619,0.1226
4,wind_gust,9231,60.9650
5,relative_humidity,23638,0.0423
6,ceiling_height,23573,0.3172
7,pressure_3hr_change,23512,0.5751


Findings\. Metric completeness at the station\-bucket grain is very strong for the core weather fields\. Temperature, precipitation, visibility, wind speed, relative humidity, and ceiling height are effectively complete after the hour\-to\-bucket rollup, while pressure change also ends up much more available here than it looked at the station\-hour grain\. Wind gust remains the clear sparse exception, which is consistent with everything we have seen so far: it is still worth keeping as a secondary contextual field, but not as a uniformly populated core weather signal\.

This last block gives us a simple shape check on the finished station\-bucket table\. We want to see the row count, station count, date range, and how many temporal buckets actually made it through so we can sanity\-check the handoff into the spatial section\.

In [24]:
# ---------------------------------------------------------------------
# Review station-bucket output shape
# ---------------------------------------------------------------------

weather_station_bucket_shape_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "station_count",
            "min_date",
            "max_date",
            "distinct_dates",
            "distinct_temporal_buckets",
        ],
        "value": [
            int(len(weather_station_temporal_bucket_df)),
            int(weather_station_temporal_bucket_df["weather_station_id"].nunique()),
            weather_station_temporal_bucket_df["date"].min(),
            weather_station_temporal_bucket_df["date"].max(),
            int(weather_station_temporal_bucket_df["date"].nunique()),
            int(weather_station_temporal_bucket_df["temporal_bucket"].nunique()),
        ],
    }
)

display(weather_station_bucket_shape_df)

,metric,value
0,row_count,23648
1,station_count,4
2,min_date,2023-01-01 00:00:00
3,max_date,2026-03-31 00:00:00
4,distinct_dates,1183
5,distinct_temporal_buckets,10


Findings\. The final shape check reinforces that this is now a coherent handoff table\. We have 23,648 rows across four stations, 1,183 distinct study dates, and all 10 temporal buckets represented in the output\. That gives us a finished station\-temporal weather layer with the right scope and coverage to move into the spatial\-assignment chapter next\.

### Section Summary

> This section took weather from a messy but usable station\-level source into a clean temporal product the rest of the project can actually build on\. We kept the approved metric set, applied the one clearly justified cleanup rule, tested bucket readiness, and then stopped when the diagnostics showed that direct bucket aggregation would overweight hours with repeated subhour reports\. From there, we introduced the canonical weather\_station\_id x observation\_hour layer, validated that it behaved like a proper intermediate contract, and only then rolled it up to the final weather\_station\_id x date x temporal\_bucket table\. By the end of the section, we have a temporally harmonized weather layer with the right grain, clean keys, full study\-period coverage, and a clear understanding of which fields are core versus secondary as we move into spatial assignment\.

## 1\.2\.5\.3 Evaluate spatial weather assignment options and build Taxi Zone weather assignment layers

This chapter brings the cleaned weather layer into contact with geography and forces us to resolve the main open question we deferred earlier: how, if at all, should station\-based weather be assigned to Taxi Zones? We start by loading the finished station\-temporal weather table, the canonical Taxi Zone geography, and a real station reference built from persisted weather source files\. Then we compare spatial assignment options directly, including a focused test of whether Central Park behaves differently enough from the airport stations that a naive nearest\-station mapping would feel misleading\.

### Load station\-temporal weather and Taxi Zone reference geography

This section loads the finished station\-temporal weather table, the canonical Taxi Zone layer, and the weather\-station point reference we need for spatial testing\. The goal is to make sure the spatial chapter starts from real persisted artifacts rather than notebook memory or guessed paths\.

First we load the weather and Taxi Zone inputs and rebuild a small station reference from the raw NOAA station\-year files\. That gives us actual station coordinates we can trust when we test nearest\-station assignment\.

In [25]:
# ---------------------------------------------------------------------
# Load station-temporal weather, Taxi Zones, and weather station reference
# ---------------------------------------------------------------------

TAXI_ZONES_PATH = PIPELINE_DATA_DIR / "1.1.6.nyc_taxi_zones.parquet"
WEATHER_RAW_STATION_YEAR_GLOB = str(
    Path("source_data") / "weather" / "noaa" / "ghcnh" / "station_year" / "*.psv"
)

if "weather_station_temporal_bucket_df" not in globals():
    weather_station_temporal_bucket_df = pd.read_parquet(
        WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH
    )

weather_station_names_df = (
    weather_station_temporal_bucket_df[
        ["weather_station_id", "weather_station_name"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

weather_station_reference_raw_df = duckdb.sql(f"""
    SELECT
        STATION AS weather_station_id,
        MAX(Station_name) AS source_station_name,
        AVG(LATITUDE) AS station_latitude,
        AVG(LONGITUDE) AS station_longitude,
        COUNT(*) AS raw_source_rows
    FROM read_csv_auto(
        '{WEATHER_RAW_STATION_YEAR_GLOB}',
        delim='|',
        header=TRUE,
        ignore_errors=TRUE
    )
    GROUP BY STATION
""").df()

weather_station_reference_df = (
    weather_station_names_df.merge(
        weather_station_reference_raw_df,
        on="weather_station_id",
        how="left",
    )
    .sort_values("weather_station_id")
    .reset_index(drop=True)
)

weather_station_reference_gdf = gpd.GeoDataFrame(
    weather_station_reference_df.copy(),
    geometry=gpd.points_from_xy(
        weather_station_reference_df["station_longitude"],
        weather_station_reference_df["station_latitude"],
    ),
    crs="EPSG:4326",
)

taxi_zones_gdf = gpd.read_parquet(TAXI_ZONES_PATH)

# Some Taxi Zones are stored as multiple polygon-part rows. Build one canonical
# geography row per location_id before any centroid or distance logic.
taxi_zone_reference_gdf = (
    taxi_zones_gdf[
        ["location_id", "zone", "borough", "geometry"]
    ]
    .dissolve(by=["location_id", "zone", "borough"], as_index=False)
    .sort_values("location_id")
    .reset_index(drop=True)
)

weather_spatial_input_summary_df = pd.DataFrame(
    {
        "metric": [
            "station_temporal_rows",
            "weather_station_count",
            "raw_taxi_zone_geometry_rows",
            "canonical_taxi_zone_rows",
            "station_reference_rows",
            "station_reference_missing_coordinate_rows",
        ],
        "value": [
            int(len(weather_station_temporal_bucket_df)),
            int(weather_station_temporal_bucket_df["weather_station_id"].nunique()),
            int(len(taxi_zones_gdf)),
            int(len(taxi_zone_reference_gdf)),
            int(len(weather_station_reference_df)),
            int(
                weather_station_reference_df[
                    ["station_latitude", "station_longitude"]
                ].isna().any(axis=1).sum()
            ),
        ],
    }
)

display(weather_spatial_input_summary_df)

,metric,value
0,station_temporal_rows,23648
1,weather_station_count,4
2,raw_taxi_zone_geometry_rows,263
3,canonical_taxi_zone_rows,260
4,station_reference_rows,4
5,station_reference_missing_coordinate_rows,0


Next we sanity\-check the station reference itself\. We mainly want to see the four station IDs, names, and coordinates we’ll use for the spatial comparison work\.

In [26]:
# ---------------------------------------------------------------------
# Review weather station reference geometry
# ---------------------------------------------------------------------

weather_station_reference_review_df = weather_station_reference_df[
    [
        "weather_station_id",
        "weather_station_name",
        "source_station_name",
        "station_latitude",
        "station_longitude",
        "raw_source_rows",
    ]
].copy()

weather_station_reference_review_df[["station_latitude", "station_longitude"]] = (
    weather_station_reference_review_df[["station_latitude", "station_longitude"]]
    .round(3)
)

display(weather_station_reference_review_df)

,weather_station_id,weather_station_name,source_station_name,station_latitude,station_longitude,raw_source_rows
0,USW00014732,New York LaGuardia Airport,LAGUARDIA AP,40.779,-73.880,77457
1,USW00014734,Newark Liberty International Airport,NEWARK LIBERTY INTL AP,40.683,-74.169,88175
2,USW00094728,NY CITY CNTRL PARK,NY CITY CNTRL PARK,40.779,-73.969,81328
3,USW00094789,New York JFK Airport,JFK INTL AP,40.639,-73.764,71389


Findings\. The rebuilt station reference looks clean and trustworthy enough for spatial work\. All four weather stations resolved with usable coordinates, the names line up with the stations we carried through the temporal workflow, and the raw source row counts match the scale we would expect from the staged weather load\. That gives us a real persisted station\-point reference to use for distance\-based assignment, rather than relying on notebook\-memory discovery tables\.

### Compare spatial assignment options and test the Central Park concern

This section tests whether a simple nearest\-station assignment is actually believable\. We compare the station\-level weather behavior directly, then look at what a naive nearest\-station lookup would do to Taxi Zones, with special attention to whether Central Park looks meaningfully different from the airport stations\.

First we compare the four stations directly at the station\-bucket grain\. This gives us a quick read on whether Central Park is moving with the airport stations or behaving like a meaningfully different local weather regime\.

In [27]:
# ---------------------------------------------------------------------
# Review station-level weather summaries at the station-bucket grain
# ---------------------------------------------------------------------

spatial_comparison_metrics = [
    "temperature",
    "precipitation",
    "visibility",
    "wind_speed",
    "relative_humidity",
]

weather_station_spatial_summary_df = (
    weather_station_temporal_bucket_df.groupby(
        ["weather_station_id", "weather_station_name"],
        dropna=False,
    )
    .agg(
        bucket_rows=("date", "size"),
        temperature_mean=("temperature", "mean"),
        precipitation_mean=("precipitation", "mean"),
        visibility_mean=("visibility", "mean"),
        wind_speed_mean=("wind_speed", "mean"),
        relative_humidity_mean=("relative_humidity", "mean"),
    )
    .reset_index()
    .sort_values("weather_station_id")
    .reset_index(drop=True)
)

weather_station_spatial_summary_df[
    [
        "temperature_mean",
        "precipitation_mean",
        "visibility_mean",
        "wind_speed_mean",
        "relative_humidity_mean",
    ]
] = weather_station_spatial_summary_df[
    [
        "temperature_mean",
        "precipitation_mean",
        "visibility_mean",
        "wind_speed_mean",
        "relative_humidity_mean",
    ]
].round(3)

display(weather_station_spatial_summary_df)

,weather_station_id,weather_station_name,bucket_rows,temperature_mean,precipitation_mean,visibility_mean,wind_speed_mean,relative_humidity_mean
0,USW00014732,New York LaGuardia Airport,5912,13.311,1.051,14.967,4.795,57.226
1,USW00014734,Newark Liberty International Airport,5912,13.737,0.897,14.930,4.220,58.695
2,USW00094728,NY CITY CNTRL PARK,5912,13.061,1.635,14.591,2.276,59.219
3,USW00094789,New York JFK Airport,5912,12.624,0.831,14.841,5.120,67.309


Findings\. At a high level, Central Park does look a bit different from the airport stations, but not so different that it immediately breaks the idea of spatial assignment\. Temperature and visibility means are all in a pretty tight band across the four stations, while the more noticeable separation shows up in precipitation, wind speed, and humidity\. Central Park stands out as calmer and wetter on average, and JFK stands out as the most humid and among the windiest, which is exactly the kind of pattern that makes this chapter necessary: there is real local variation here, not just interchangeable station readings\.

Next we test the Central Park concern more directly by lining its weather up against each airport station on the same date and temporal bucket\. We are looking for whether the relationships still feel tight enough to support a single nearest\-station assignment story\.

In [28]:
# ---------------------------------------------------------------------
# Compare Central Park against airport stations on aligned buckets
# ---------------------------------------------------------------------

central_park_station_id = "USW00094728"
comparison_metrics = [
    "temperature",
    "precipitation",
    "visibility",
    "wind_speed",
    "relative_humidity",
]

central_park_bucket_df = (
    weather_station_temporal_bucket_df[
        weather_station_temporal_bucket_df["weather_station_id"] == central_park_station_id
    ]
    [["date", "temporal_bucket", *comparison_metrics]]
    .rename(
        columns={metric: f"central_park_{metric}" for metric in comparison_metrics}
    )
    .reset_index(drop=True)
)

comparison_rows = []
airport_station_ids = [
    station_id
    for station_id in weather_station_temporal_bucket_df["weather_station_id"].dropna().unique()
    if station_id != central_park_station_id
]

for airport_station_id in sorted(airport_station_ids):
    airport_station_name = weather_station_temporal_bucket_df.loc[
        weather_station_temporal_bucket_df["weather_station_id"] == airport_station_id,
        "weather_station_name",
    ].iloc[0]

    airport_bucket_df = (
        weather_station_temporal_bucket_df[
            weather_station_temporal_bucket_df["weather_station_id"] == airport_station_id
        ]
        [["date", "temporal_bucket", *comparison_metrics]]
        .rename(
            columns={metric: f"airport_{metric}" for metric in comparison_metrics}
        )
        .reset_index(drop=True)
    )

    aligned_bucket_df = central_park_bucket_df.merge(
        airport_bucket_df,
        on=["date", "temporal_bucket"],
        how="inner",
    )

    for metric in comparison_metrics:
        metric_pair_df = aligned_bucket_df[
            [f"central_park_{metric}", f"airport_{metric}"]
        ].dropna()

        matched_rows = len(metric_pair_df)
        correlation = (
            metric_pair_df[f"central_park_{metric}"].corr(metric_pair_df[f"airport_{metric}"])
            if matched_rows >= 2
            else np.nan
        )
        mean_abs_diff = (
            (metric_pair_df[f"central_park_{metric}"] - metric_pair_df[f"airport_{metric}"])
            .abs()
            .mean()
            if matched_rows > 0
            else np.nan
        )

        comparison_rows.append(
            {
                "airport_station_id": airport_station_id,
                "airport_station_name": airport_station_name,
                "metric": metric,
                "matched_bucket_rows": int(matched_rows),
                "correlation": correlation,
                "mean_abs_difference": mean_abs_diff,
            }
        )

central_park_airport_comparison_df = pd.DataFrame(comparison_rows)
central_park_airport_comparison_df[["correlation", "mean_abs_difference"]] = (
    central_park_airport_comparison_df[["correlation", "mean_abs_difference"]].round(3)
)

display(central_park_airport_comparison_df)

,airport_station_id,airport_station_name,metric,matched_bucket_rows,correlation,mean_abs_difference
0,USW00014732,New York LaGuardia Airport,temperature,5907,0.996,0.678
1,USW00014732,New York LaGuardia Airport,precipitation,5912,0.827,0.867
2,USW00014732,New York LaGuardia Airport,visibility,5901,0.866,0.695
3,USW00014732,New York LaGuardia Airport,wind_speed,5886,0.799,2.525
4,USW00014732,New York LaGuardia Airport,relative_humidity,5905,0.969,3.912
5,USW00014734,Newark Liberty International Airport,temperature,5907,0.995,1.024
6,USW00014734,Newark Liberty International Airport,precipitation,5912,0.702,1.064
7,USW00014734,Newark Liberty International Airport,visibility,5901,0.872,0.666
8,USW00014734,Newark Liberty International Airport,wind_speed,5886,0.746,2.014
9,USW00014734,Newark Liberty International Airport,relative_humidity,5905,0.961,3.841


Findings\. The aligned bucket comparison sharpens that picture\. Central Park tracks the airport stations very closely on temperature and relative humidity, with especially strong temperature correlations across all three comparisons\. Visibility also hangs together reasonably well\. The weaker relationships are precipitation and wind speed, where the correlations are noticeably lower and the average differences are bigger, especially against JFK\. So the story is not that Central Park is a completely separate weather world; it is that some fields move together strongly across the region, while others retain enough local variation that a nearest\-station assignment should be treated as a real modeling choice rather than an automatic truth\.

Last, we look at what a naive nearest\-station assignment would actually do to Taxi Zones\. This gives us a concrete benchmark before we decide whether nearest\-station mapping is good enough or whether we need a different weather\-to\-zone interpretation\.

In [29]:
# ---------------------------------------------------------------------
# Preview nearest-station Taxi Zone assignment
# ---------------------------------------------------------------------

taxi_zone_reference_projected_gdf = taxi_zone_reference_gdf.to_crs("EPSG:2263").copy()
taxi_zone_reference_projected_gdf["zone_centroid_geometry"] = (
    taxi_zone_reference_projected_gdf.geometry.centroid
)

weather_station_reference_projected_gdf = weather_station_reference_gdf.to_crs("EPSG:2263").copy()

nearest_station_rows = []
for _, zone_row in taxi_zone_reference_projected_gdf.iterrows():
    centroid_geometry = zone_row["zone_centroid_geometry"]
    station_distance_df = weather_station_reference_projected_gdf[
        [
            "weather_station_id",
            "weather_station_name",
            "geometry",
        ]
    ].copy()
    station_distance_df["distance_feet"] = station_distance_df.geometry.distance(centroid_geometry)

    nearest_station_row = station_distance_df.sort_values("distance_feet").iloc[0]

    nearest_station_rows.append(
        {
            "taxi_zone_id": zone_row.get("location_id"),
            "taxi_zone_name": zone_row.get("zone"),
            "taxi_zone_borough": zone_row.get("borough"),
            "nearest_weather_station_id": nearest_station_row["weather_station_id"],
            "nearest_weather_station_name": nearest_station_row["weather_station_name"],
            "distance_feet": float(nearest_station_row["distance_feet"]),
            "distance_miles": float(nearest_station_row["distance_feet"] / 5280),
        }
    )

nearest_station_lookup_preview_df = pd.DataFrame(nearest_station_rows)

nearest_station_lookup_summary_df = (
    nearest_station_lookup_preview_df.groupby(
        ["nearest_weather_station_id", "nearest_weather_station_name"],
        dropna=False,
    )
    .agg(
        taxi_zone_count=("taxi_zone_id", "size"),
        avg_distance_miles=("distance_miles", "mean"),
        max_distance_miles=("distance_miles", "max"),
    )
    .reset_index()
    .sort_values("taxi_zone_count", ascending=False)
    .reset_index(drop=True)
)

nearest_station_lookup_summary_df[["avg_distance_miles", "max_distance_miles"]] = (
    nearest_station_lookup_summary_df[["avg_distance_miles", "max_distance_miles"]].round(3)
)

display(nearest_station_lookup_summary_df)

,nearest_weather_station_id,nearest_weather_station_name,taxi_zone_count,avg_distance_miles,max_distance_miles
0,USW00094728,NY CITY CNTRL PARK,99,4.047,9.739
1,USW00014732,New York LaGuardia Airport,86,4.651,8.406
2,USW00094789,New York JFK Airport,44,6.125,11.128
3,USW00014734,Newark Liberty International Airport,31,8.031,12.005


Findings\. The nearest\-station preview shows that a simple centroid\-based assignment would give Central Park the largest share of Taxi Zones by a wide margin, followed by LaGuardia, with JFK and Newark covering much smaller portions of the city\. Average centroid\-to\-station distances are still fairly modest, but some maximum distances push well past 9 to 12 miles, which is not trivial for local weather context\. That means a nearest\-station lookup is operationally very feasible, but it would also bake in a strong spatial interpretation: a lot of the city would inherit Central Park weather by default\. That is exactly the decision we now need to make explicitly\.

A quick map helps us sanity\-check the nearest\-station assignment in a way the tables cannot\. We want to see whether the assigned station footprints look geographically coherent across the Taxi Zone layer before we decide whether this is believable enough to use downstream\.

In [30]:
# ---------------------------------------------------------------------
# Visual QA: nearest-station Taxi Zone assignment choropleth
# ---------------------------------------------------------------------

import json
import plotly.graph_objects as go

from project_branding import BRAND_COLORS


if not hasattr(go, "Choroplethmap"):
    raise RuntimeError(
        "This map requires Plotly's MapLibre-backed go.Choroplethmap. "
        "Do not fall back to token-dependent Choroplethmapbox."
    )


nearest_station_zone_map_gdf = taxi_zone_reference_gdf.merge(
    nearest_station_lookup_preview_df,
    left_on="location_id",
    right_on="taxi_zone_id",
    how="left",
)

nearest_station_zone_map_gdf = gpd.GeoDataFrame(
    nearest_station_zone_map_gdf,
    geometry="geometry",
    crs=taxi_zone_reference_gdf.crs,
).to_crs("EPSG:4326")


# ---------------------------------------------------------------------
# Build the portable polygon scaffold
# ---------------------------------------------------------------------

nearest_station_plot_gdf = (
    nearest_station_zone_map_gdf
    .to_crs("EPSG:2263")
    .copy()
)

nearest_station_plot_gdf["geometry"] = (
    nearest_station_plot_gdf["geometry"]
    .simplify(
        tolerance=75,
        preserve_topology=True,
    )
)

nearest_station_plot_gdf = (
    nearest_station_plot_gdf
    .to_crs("EPSG:4326")
    .reset_index(drop=True)
)

nearest_station_plot_gdf["location_id"] = (
    pd.to_numeric(
        nearest_station_plot_gdf["location_id"],
        errors="raise",
    )
    .astype("Int64")
    .astype(str)
)

station_labels = sorted(
    nearest_station_plot_gdf[
        "nearest_weather_station_name"
    ]
    .dropna()
    .astype(str)
    .unique()
)

station_code_map = {
    station_name: station_code
    for station_code, station_name in enumerate(station_labels)
}

nearest_station_plot_gdf["station_code"] = (
    nearest_station_plot_gdf["nearest_weather_station_name"]
    .astype(str)
    .map(station_code_map)
)

station_colors = [
    BRAND_COLORS["dark_teal"],
    BRAND_COLORS["terracotta"],
    BRAND_COLORS["seafoam"],
    BRAND_COLORS["pale_peach"],
]

discrete_colorscale = []

for color_index, _ in enumerate(station_labels):
    color = station_colors[color_index % len(station_colors)]
    start = color_index / max(len(station_labels), 1)
    end = (color_index + 1) / max(len(station_labels), 1)

    discrete_colorscale.extend([
        [start, color],
        [end, color],
    ])

nearest_station_geojson = json.loads(
    nearest_station_plot_gdf[
        [
            "location_id",
            "zone",
            "borough",
            "geometry",
        ]
    ].to_json()
)


# ---------------------------------------------------------------------
# Render portable MapLibre choropleth
# ---------------------------------------------------------------------

nearest_station_map_fig = go.Figure(
    go.Choroplethmap(
        geojson=nearest_station_geojson,
        locations=nearest_station_plot_gdf["location_id"],
        z=nearest_station_plot_gdf["station_code"],
        featureidkey="properties.location_id",
        colorscale=discrete_colorscale,
        zmin=0,
        zmax=max(len(station_labels) - 1, 1),
        marker={
            "opacity": 0.78,
            "line": {
                "width": 0.6,
                "color": BRAND_COLORS["dark_teal"],
            },
        },
        colorbar={
            "title": {"text": "Nearest station"},
            "tickvals": list(range(len(station_labels))),
            "ticktext": station_labels,
        },
        customdata=np.column_stack(
            [
                nearest_station_plot_gdf["zone"],
                nearest_station_plot_gdf["borough"],
                nearest_station_plot_gdf[
                    "nearest_weather_station_name"
                ],
                nearest_station_plot_gdf["distance_miles"],
            ]
        ),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Borough: %{customdata[1]}<br>"
            "Nearest weather station: %{customdata[2]}<br>"
            "Distance: %{customdata[3]:.2f} miles"
            "<extra></extra>"
        ),
    )
)

nearest_station_map_fig.update_layout(
    title="Nearest Weather Station by Taxi Zone",
    map={
        "style": "carto-positron",
        "center": {"lat": 40.71, "lon": -73.90},
        "zoom": 9.15,
    },
    margin={"l": 20, "r": 20, "t": 60, "b": 20},
    height=600,
)

nearest_station_map_fig.show()

Findings\. The choropleth makes the nearest\-station assignment look geographically coherent rather than arbitrary\. The resulting station footprints are contiguous and intuitive: Newark mainly covers Staten Island and the western edge, JFK takes the southern and southeastern zones, LaGuardia covers much of Queens and the Bronx, and Central Park anchors a dense Manhattan\-centered core\. In other words, the map does not show noisy patchwork or implausible leapfrogging across the city; it shows a clean spatial partition that is easy to explain and audit\.

> ⚖️ Decision\. We will use a distance\-weighted multi\-station weather assignment as the Taxi Zone weather handoff, rather than a hard nearest\-station lookup\. The Central Park comparison showed that the stations move together enough for a shared regional weather signal, while the map also made clear that many zones sit between plausible station influences rather than cleanly belonging to just one\. With only four stations, a distance\-weighted blend stays simple to compute, is still easy to explain, and better reflects gradual spatial transitions across the city than a winner\-take\-all assignment\.

### Build Taxi Zone distance\-weighted weather assignment

This subsection turns the spatial decision into a concrete assignment layer\. We are no longer asking whether weather can be mapped to Taxi Zones; we are building the chosen distance\-weighted handoff itself so downstream notebooks can join weather context at the same \`taxi\_zone\_id x date x temporal\_bucket\` rhythm as the rest of the mobility foundation\.

We start by measuring each Taxi Zone centroid against all four stations and converting those distances into normalized inverse\-distance weights\. The point is to make every zone’s blend explicit and auditable rather than hiding the assignment logic inside a later merge\.

In [31]:
# ---------------------------------------------------------------------
# Build Taxi Zone distance-weighted weather assignment
# ---------------------------------------------------------------------

print_step("Using upstream Taxi Zone and station reference layers for weighted assignment.")

if should_build(
    WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH,
    REBUILD_WEATHER_ZONE_ASSIGNMENT_WEIGHTS,
):
    print_step("Building Taxi Zone distance-weighted weather assignment.")

    taxi_zone_reference_projected_gdf = taxi_zone_reference_gdf.to_crs("EPSG:2263").copy()
    taxi_zone_reference_projected_gdf["zone_centroid_geometry"] = (
        taxi_zone_reference_projected_gdf.geometry.centroid
    )

    weather_station_points_projected_gdf = weather_station_reference_gdf.to_crs("EPSG:2263").copy()

    assignment_rows = []
    for _, zone_row in taxi_zone_reference_projected_gdf.iterrows():
        centroid_geometry = zone_row["zone_centroid_geometry"]
        station_distance_df = weather_station_points_projected_gdf[
            ["weather_station_id", "weather_station_name", "geometry"]
        ].copy()
        station_distance_df["distance_feet"] = station_distance_df.geometry.distance(centroid_geometry)
        station_distance_df["distance_miles"] = station_distance_df["distance_feet"] / 5280
        station_distance_df["inverse_distance_weight_raw"] = 1 / station_distance_df[
            "distance_miles"
        ].clip(lower=WEATHER_DISTANCE_WEIGHT_EPSILON_MILES)
        station_distance_df["normalized_weight"] = (
            station_distance_df["inverse_distance_weight_raw"]
            / station_distance_df["inverse_distance_weight_raw"].sum()
        )
        station_distance_df = station_distance_df.sort_values(
            "normalized_weight", ascending=False
        ).reset_index(drop=True)
        station_distance_df["weight_rank"] = station_distance_df.index + 1

        for _, station_row in station_distance_df.iterrows():
            assignment_rows.append(
                {
                    "taxi_zone_id": zone_row["location_id"],
                    "taxi_zone_name": zone_row["zone"],
                    "taxi_zone_borough": zone_row["borough"],
                    "weather_station_id": station_row["weather_station_id"],
                    "weather_station_name": station_row["weather_station_name"],
                    "distance_feet": float(station_row["distance_feet"]),
                    "distance_miles": float(station_row["distance_miles"]),
                    "inverse_distance_weight_raw": float(
                        station_row["inverse_distance_weight_raw"]
                    ),
                    "normalized_weight": float(station_row["normalized_weight"]),
                    "weight_rank": int(station_row["weight_rank"]),
                    "weather_assignment_method": "inverse_distance_centroid_weighting",
                }
            )

    taxi_zone_weather_assignment_weights_df = pd.DataFrame(assignment_rows)
    taxi_zone_weather_assignment_weights_df.to_parquet(
        WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH,
        index=False,
    )
    print_step(
        f"Wrote Taxi Zone weather assignment weights: {WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH}"
    )
else:
    print_step("Loading existing Taxi Zone weather assignment weights.")
    taxi_zone_weather_assignment_weights_df = pd.read_parquet(
        WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH
    )

zone_weight_sum_check = (
    taxi_zone_weather_assignment_weights_df.groupby("taxi_zone_id", dropna=False)[
        "normalized_weight"
    ]
    .sum()
)

weather_assignment_build_summary_df = pd.DataFrame(
    {
        "metric": [
            "zone_station_weight_rows",
            "taxi_zone_count",
            "station_count",
            "min_distance_miles",
            "max_distance_miles",
            "min_zone_weight_sum",
            "max_zone_weight_sum",
        ],
        "value": [
            int(len(taxi_zone_weather_assignment_weights_df)),
            int(taxi_zone_weather_assignment_weights_df["taxi_zone_id"].nunique()),
            int(taxi_zone_weather_assignment_weights_df["weather_station_id"].nunique()),
            round(float(taxi_zone_weather_assignment_weights_df["distance_miles"].min()), 3),
            round(float(taxi_zone_weather_assignment_weights_df["distance_miles"].max()), 3),
            round(float(zone_weight_sum_check.min()), 3),
            round(float(zone_weight_sum_check.max()), 3),
        ],
    }
)

display(weather_assignment_build_summary_df)

[1.2.5] Using upstream Taxi Zone and station reference layers for weighted assignment.
[1.2.5] Loading existing Taxi Zone weather assignment weights.


,metric,value
0,zone_station_weight_rows,1040.000
1,taxi_zone_count,260.000
2,station_count,4.000
3,min_distance_miles,0.312
4,max_distance_miles,25.922
5,min_zone_weight_sum,1.000
6,max_zone_weight_sum,1.000


Findings\. The rebuilt weight table looks structurally right\. We have exactly 1,040 taxi\_zone x station rows, which is the expected 260 x 4 shape, and every zone is drawing from the full four\-station set\. The distance range is still fairly broad, from about 0\.312 miles up to 25\.922 miles, but the important part is that the row count and weight math  line up with the intended assignment design\.

Then we sanity\-check the weight table itself\. We want to confirm the key shape is right, every zone has a full four\-station blend, and the normalized weights behave exactly the way the assignment rule says they should\.

In [32]:
# ---------------------------------------------------------------------
# Validate Taxi Zone weather assignment weights
# ---------------------------------------------------------------------

zone_weight_counts = taxi_zone_weather_assignment_weights_df.groupby(
    "taxi_zone_id",
    dropna=False,
).size()
zone_weight_sums = taxi_zone_weather_assignment_weights_df.groupby(
    "taxi_zone_id",
    dropna=False,
)["normalized_weight"].sum()

weather_assignment_weight_validation_df = pd.DataFrame(
    {
        "check": [
            "unique_zone_station_key",
            "non_null_taxi_zone_id",
            "non_null_weather_station_id",
            "four_station_blend_per_zone",
            "zone_weights_sum_to_one",
            "positive_row_count",
        ],
        "status": [
            not taxi_zone_weather_assignment_weights_df.duplicated(
                ["taxi_zone_id", "weather_station_id"]
            ).any(),
            taxi_zone_weather_assignment_weights_df["taxi_zone_id"].notna().all(),
            taxi_zone_weather_assignment_weights_df["weather_station_id"].notna().all(),
            bool((zone_weight_counts == 4).all()),
            bool(np.isclose(zone_weight_sums, 1.0, atol=1e-9).all()),
            len(taxi_zone_weather_assignment_weights_df) > 0,
        ],
    }
)

display(weather_assignment_weight_validation_df)

,check,status
0,unique_zone_station_key,True
1,non_null_taxi_zone_id,True
2,non_null_weather_station_id,True
3,four_station_blend_per_zone,True
4,zone_weights_sum_to_one,True
5,positive_row_count,True


Findings\. The validation block confirms that the weighted assignment artifact is now clean and usable\. The taxi\_zone\_id x weather\_station\_id key is unique, all required IDs are populated, every Taxi Zone has a complete four\-station blend, and every zone’s normalized weights sum to exactly 1\. At this point the spatial assignment layer looks ready to serve as the upstream input for the final Taxi Zone weather context build\.

Last we look at how the blends actually distribute across the city\. We are checking whether this behaves like a smooth multi\-station surface or quietly collapses back into a winner\-take\-all assignment for most zones\.

In [33]:
# ---------------------------------------------------------------------
# Review Taxi Zone weather weight mix
# ---------------------------------------------------------------------

zone_top_weight_df = (
    taxi_zone_weather_assignment_weights_df.sort_values(
        ["taxi_zone_id", "normalized_weight"],
        ascending=[True, False],
    )
    .groupby("taxi_zone_id", dropna=False)
    .head(2)
    .copy()
)

zone_top_weight_rank_df = (
    zone_top_weight_df.pivot_table(
        index=["taxi_zone_id"],
        columns="weight_rank",
        values="normalized_weight",
        aggfunc="first",
    )
    .rename(columns={1: "top_weight", 2: "second_weight"})
    .reset_index()
)

dominant_station_weight_mix_df = (
    taxi_zone_weather_assignment_weights_df[
        taxi_zone_weather_assignment_weights_df["weight_rank"] == 1
    ]
    .merge(zone_top_weight_rank_df, on="taxi_zone_id", how="left")
    .groupby(["weather_station_id", "weather_station_name"], dropna=False)
    .agg(
        dominant_zone_count=("taxi_zone_id", "size"),
        avg_top_weight=("top_weight", "mean"),
        avg_second_weight=("second_weight", "mean"),
        max_top_weight=("top_weight", "max"),
    )
    .reset_index()
    .sort_values("dominant_zone_count", ascending=False)
    .reset_index(drop=True)
)

dominant_station_weight_mix_df[[
    "avg_top_weight",
    "avg_second_weight",
    "max_top_weight",
]] = dominant_station_weight_mix_df[[
    "avg_top_weight",
    "avg_second_weight",
    "max_top_weight",
]].round(3)

display(dominant_station_weight_mix_df)

,weather_station_id,weather_station_name,dominant_zone_count,avg_top_weight,avg_second_weight,max_top_weight
0,USW00094728,NY CITY CNTRL PARK,99,0.477,0.248,0.896
1,USW00014732,New York LaGuardia Airport,86,0.448,0.291,0.853
2,USW00094789,New York JFK Airport,44,0.430,0.236,0.776
3,USW00014734,Newark Liberty International Airport,31,0.420,0.217,0.888


Findings\. The dominance mix behaves like a real weighted blend rather than a disguised nearest\-station lookup\. Central Park remains the top\-contributing station for the most zones, followed by LaGuardia, JFK, and Newark, but the average top weight only ranges from about 0\.420 to 0\.477, with the second\-largest station still carrying meaningful share across the board\. So the final pattern feels appropriately blended: each zone has a leading station influence, but most zones are still pulling real signal from multiple stations rather than inheriting weather from a single winner\-take\-all source\.

## 1\.2\.5\.4 Validate and export

This final chapter keeps the handoff simple\. We turn the weighted station assignment into the finished Taxi Zone weather context table, run a compact contract check on the result, and write the export artifacts that downstream notebooks can consume without redoing any spatial logic\.

### Export weighted Taxi Zone weather artifacts

We begin by applying the assignment weights to the station\-bucket weather table so we can produce the final Taxi Zone weather context at the project’s shared grain\. Each retained weather metric will use a weighted average across the available stations for that zone\-date\-bucket combination\.

In [34]:
# ---------------------------------------------------------------------
# Build Taxi Zone weather context output
# ---------------------------------------------------------------------

print_step("Loading station-bucket weather table and Taxi Zone assignment weights.")
weather_station_temporal_bucket_df = pd.read_parquet(
    WEATHER_STATION_TEMPORAL_BUCKET_OUTPUT_PATH
)
taxi_zone_weather_assignment_weights_df = pd.read_parquet(
    WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH
)

if should_build(WEATHER_ZONE_CONTEXT_OUTPUT_PATH, REBUILD_WEATHER_ZONE_CONTEXT):
    print_step("Building Taxi Zone weather context output.")

    station_bucket_with_weights_df = taxi_zone_weather_assignment_weights_df.merge(
        weather_station_temporal_bucket_df,
        on=["weather_station_id", "weather_station_name"],
        how="left",
    )

    zone_context_key_columns = [
        "taxi_zone_id",
        "taxi_zone_name",
        "taxi_zone_borough",
        "date",
        "year",
        "month",
        "day_of_week",
        "daypart",
        "daypart_order",
        "temporal_bucket",
        "temporal_bucket_order",
        "observation_sequence_id",
        "pre_post_cp",
    ]

    weather_zone_context_df = (
        station_bucket_with_weights_df[zone_context_key_columns]
        .drop_duplicates()
        .sort_values(["taxi_zone_id", "date", "temporal_bucket_order"])
        .reset_index(drop=True)
    )

    for metric in RETAINED_WEATHER_MEASURE_COLUMNS:
        metric_component_df = station_bucket_with_weights_df[
            zone_context_key_columns + [metric, "normalized_weight"]
        ].copy()
        metric_component_df = metric_component_df[metric_component_df[metric].notna()].copy()
        metric_component_df["metric_weighted_value"] = (
            metric_component_df[metric] * metric_component_df["normalized_weight"]
        )

        # Re-normalize over the stations that actually reported a value for this
        # metric in this zone-date-bucket so sparse fields do not get biased low.
        metric_weighted_df = (
            metric_component_df.groupby(zone_context_key_columns, dropna=False)
            .agg(
                metric_weight_sum=("normalized_weight", "sum"),
                metric_weighted_sum=("metric_weighted_value", "sum"),
            )
            .reset_index()
        )
        metric_weighted_df[metric] = (
            metric_weighted_df["metric_weighted_sum"]
            / metric_weighted_df["metric_weight_sum"]
        )
        metric_weighted_df = metric_weighted_df[
            zone_context_key_columns + [metric]
        ].copy()

        weather_zone_context_df = weather_zone_context_df.merge(
            metric_weighted_df,
            on=zone_context_key_columns,
            how="left",
        )

    dominant_station_lookup_df = taxi_zone_weather_assignment_weights_df[
        taxi_zone_weather_assignment_weights_df["weight_rank"] == 1
    ][
        [
            "taxi_zone_id",
            "weather_station_id",
            "weather_station_name",
            "normalized_weight",
        ]
    ].rename(
        columns={
            "weather_station_id": "dominant_weather_station_id",
            "weather_station_name": "dominant_weather_station_name",
            "normalized_weight": "dominant_weather_station_weight",
        }
    )

    weather_zone_context_df = weather_zone_context_df.merge(
        dominant_station_lookup_df,
        on="taxi_zone_id",
        how="left",
    )
    weather_zone_context_df["weather_assignment_method"] = (
        "inverse_distance_centroid_weighting"
    )
    weather_zone_context_df["weather_station_blend_count"] = 4

    weather_zone_context_df.to_parquet(
        WEATHER_ZONE_CONTEXT_OUTPUT_PATH,
        index=False,
    )
    print_step(f"Wrote Taxi Zone weather context: {WEATHER_ZONE_CONTEXT_OUTPUT_PATH}")
else:
    print_step("Loading existing Taxi Zone weather context output.")
    weather_zone_context_df = pd.read_parquet(WEATHER_ZONE_CONTEXT_OUTPUT_PATH)

weather_zone_context_summary_df = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "taxi_zone_count",
            "min_date",
            "max_date",
            "distinct_dates",
            "distinct_temporal_buckets",
        ],
        "value": [
            int(len(weather_zone_context_df)),
            int(weather_zone_context_df["taxi_zone_id"].nunique()),
            weather_zone_context_df["date"].min(),
            weather_zone_context_df["date"].max(),
            int(weather_zone_context_df["date"].nunique()),
            int(weather_zone_context_df["temporal_bucket"].nunique()),
        ],
    }
)

display(weather_zone_context_summary_df)

[1.2.5] Loading station-bucket weather table and Taxi Zone assignment weights.
[1.2.5] Loading existing Taxi Zone weather context output.


,metric,value
0,row_count,1537120
1,taxi_zone_count,260
2,min_date,2023-01-01 00:00:00
3,max_date,2026-03-31 00:00:00
4,distinct_dates,1183
5,distinct_temporal_buckets,10


Findings\. The final Taxi Zone weather context output has the shape we wanted\. We now have 1,537,120 rows covering all 260 Taxi Zones across the full study window from January 1, 2023 through March 31, 2026, with all 10 temporal buckets represented\. In other words, weather has now been successfully expanded from a four\-station temporal layer into a full zone\-level contextual layer that matches the project’s shared Taxi Zone x date x temporal bucket rhythm\.

Then we run one last contract check on the finished zone\-level table\. We want to make sure the key is unique, the shared temporal fields survived intact, and the output really is ready for downstream joins\.

In [35]:
# ---------------------------------------------------------------------
# Validate Taxi Zone weather context output
# ---------------------------------------------------------------------

weather_zone_context_validation_df = pd.DataFrame(
    {
        "check": [
            "unique_zone_date_bucket_key",
            "non_null_taxi_zone_id",
            "non_null_date",
            "non_null_temporal_bucket",
            "study_period_date_bounds",
            "positive_row_count",
        ],
        "status": [
            not weather_zone_context_df.duplicated(
                ["taxi_zone_id", "date", "temporal_bucket"]
            ).any(),
            weather_zone_context_df["taxi_zone_id"].notna().all(),
            weather_zone_context_df["date"].notna().all(),
            weather_zone_context_df["temporal_bucket"].notna().all(),
            bool(
                weather_zone_context_df["date"].between(
                    STUDY_START_DATE,
                    STUDY_END_DATE,
                ).all()
            ),
            len(weather_zone_context_df) > 0,
        ],
    }
)

display(weather_zone_context_validation_df)

,check,status
0,unique_zone_date_bucket_key,True
1,non_null_taxi_zone_id,True
2,non_null_date,True
3,non_null_temporal_bucket,True
4,study_period_date_bounds,True
5,positive_row_count,True


Findings\. The final validation check passed cleanly, which is the most important signal here\. The taxi\_zone\_id x date x temporal\_bucket key is unique, the required fields are populated, the study\-period bounds are correct, and the output contains positive row volume\. That means the zone\-level weather context is structurally ready to join downstream with the canonical mobility panels rather than still sitting in an exploratory or provisional state\.

Last we do a simple artifact check so the handoff is obvious\. This gives us one place to confirm that both the reusable weight layer and the finished Taxi Zone weather context landed where the downstream notebooks will expect them\.

In [36]:
# ---------------------------------------------------------------------
# Review exported weather handoff artifacts
# ---------------------------------------------------------------------

weather_export_artifacts_df = pd.DataFrame(
    {
        "artifact": [
            "taxi_zone_weather_assignment_weights",
            "weather_taxi_zone_context",
        ],
        "path": [
            str(WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH),
            str(WEATHER_ZONE_CONTEXT_OUTPUT_PATH),
        ],
        "exists_after_run": [
            WEATHER_ZONE_ASSIGNMENT_WEIGHTS_OUTPUT_PATH.exists(),
            WEATHER_ZONE_CONTEXT_OUTPUT_PATH.exists(),
        ],
    }
)

display(weather_export_artifacts_df)

,artifact,path,exists_after_run
0,taxi_zone_weather_assignment_weights,pipeline_data/1.2.5.harmonized_weather/taxi_zo...,True
1,weather_taxi_zone_context,pipeline_data/1.2.5.harmonized_weather/weather...,True


Findings\. The export check confirms that both handoff artifacts landed where they should\. We now have a reusable Taxi Zone weather assignment weight table and a finished Taxi Zone weather context table written to the harmonized weather directory, which gives downstream notebooks both the interpretable spatial weighting layer and the ready\-to\-join zone\-level contextual weather output\.

## Close

This notebook takes weather from a cleaned station\-level sidecar to a fully harmonized Taxi Zone context layer that the rest of the project can actually use\. We standardized the shared temporal fields, cleaned and retained the approved weather measures, resolved repeated subhour reporting through a canonical station\-hour layer, aggregated to station\-date\-bucket, tested the spatial assignment question directly, and built the final distance\-weighted Taxi Zone handoff\. Weather is no longer floating beside the mobility pipeline; it now lands in a stable, documented context table that lines up with the same study period and temporal structure as the rest of the foundation\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=5c43663c-345d-4c21-9332-dd4f928af3ba' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>